# 15. 주거·통근 인사이트 분석

이 노트북은 14_2에서 확정한 행정동 유형화 결과와 근무동–거주동 대중교통 경로 데이터를 결합하여,
**“어디가 싸다”가 아니라 주거비 절감과 통근부담의 교환관계, 실제 총부담, 근무지 효과, 개인 조건에 따른 추천 전환점**을 분석한다.

## 전체 분석 흐름

### 분석 1. 주거비를 줄이면 통근 부담이 얼마나 늘어나는가
- 근무동별 기준 거주비 설정
- 10만 원 절감당 편도 통근시간 증가
- 10만 원 절감당 월 교통비 증가
- 10만 원 절감당 통근시간비용 증가
- 실질 순절감액
- 손익분기 주거비 절감액
- 산점도 / 주거비 구간 비교 / 관계 회귀

### 분석 2. 어떤 지역이 실제로 비용 효율적인가
- 총 주거·통근 부담
- A/B/C/D 비용효율 유형
- 월세 착시 지역 / 숨은 효율 지역
- 비용 효율성 지수
- 주거비 순위 vs 총부담 순위
- 스피어만 순위상관
- 주거비·통근비용 구성비
- 소득 시나리오별 비용 효율 순위

### 분석 3. 근무지에 따라 최적 주거지가 어떻게 달라지는가
- 업무지구별 후보 필터
- 업무지구별 Top 10
- Top 10 중복률
- 동일 거주동의 근무지별 순위 변화
- 업무지구 특화도
- 권역/통근권 단위 요약
- 추천 1순위 전환 행렬

### 분석 4. 추천 결과를 결정하는 핵심 조건은 무엇인가
- 추천점수 구성
- 변수별 기여도
- 가중치 민감도
- 소득 민감도
- 예산 민감도
- 추천 1순위 전환점
- Top-K 중복률
- 스피어만 순위상관
- 평균 순위변동

## 14_2 결과 연결

최종 유형화 결과 `dong_typology_final.csv`의 다음 정보를 거주동 메타정보로 붙인다.

- 최종 6개 행정동 유형
- 주거비 × 통근부담 4사분면
- 종합부담 × 통근구조 4사분면
- 청년 1인세대 비율

유형화는 추천점수를 직접 결정하지 않고 **결과를 설명하는 메타정보**로 사용한다.

In [1]:
# =========================================================
# 1. 라이브러리 / 기본 설정
# =========================================================

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

# 프로젝트 공통 기준
WORK_DAYS_PER_MONTH = 21
MONTHLY_WORK_HOURS = 209

# 메인 분석 시간가치:
# 특정 소득을 임의로 가정하지 않고 공통 기준 사용
BASE_TIME_VALUE_PER_HOUR = 10_320
BASE_TIME_VALUE_FACTOR = 1.0

# 민감도 분석에서만 월소득 기반 시간가치를 별도로 사용
SCENARIO_BASE_MONTHLY_INCOME = 3_000_000

# 현실성 필터
MAX_ONEWAY_MINUTES = 90
BASELINE_NEARBY_MINUTES = 30
SERVICE_MAX_COMMUTE_MINUTES = 60

# '10만원 절감당' 비율이 작은 분모 때문에 폭발하지 않도록
# 최소 10만원 이상 실제 주거비가 절감되는 후보만 비율지표 계산
MIN_SAVING_FOR_100K_METRIC = 100_000

TOP_K = 10

print("월 출근일수:", WORK_DAYS_PER_MONTH)
print("월 근로시간:", MONTHLY_WORK_HOURS)
print("메인 시간가치:", f"{BASE_TIME_VALUE_PER_HOUR:,}원/시간")
print("메인 시간가치 반영계수:", BASE_TIME_VALUE_FACTOR)
print(
    "10만원 절감당 지표 최소 절감액:",
    f"{MIN_SAVING_FOR_100K_METRIC:,}원"
)


월 출근일수: 21
월 근로시간: 209
메인 시간가치: 10,320원/시간
메인 시간가치 반영계수: 1.0
10만원 절감당 지표 최소 절감액: 100,000원


## 2. 파일 경로

필수 파일은 두 종류다.

1. **14_2 최종 유형화**
   - `dong_typology_final.csv`
2. **근무동–거주동 대중교통 경로**
   - 앞 단계에서 만든 TMAP/분석 준비 CSV

경로 파일은 프로젝트 내에서 후보 파일명을 순서대로 탐색한다.

In [2]:
# =========================================================
# 2. 프로젝트 경로 / 입력 파일
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    REPO_DIR = CURRENT_DIR
else:
    raise FileNotFoundError(
        "저장소 루트 또는 notebooks 폴더에서 실행하세요.\n"
        f"현재 위치: {CURRENT_DIR}"
    )

WORKSPACE_DIR = REPO_DIR.parent
DATA_DIR = WORKSPACE_DIR / "project_data"
PROCESSED_DIR = DATA_DIR / "processed"

# 14_2 최종 유형화
TYPOLOGY_FILE = (
    PROCESSED_DIR
    / "dong_typology_final.csv"
)

# 10번 최종 OD 경로 데이터
ROUTE_FILE = (
    PROCESSED_DIR
    / "commute_routes_analysis_ready.csv"
)

# 11번 거주동별 대표 통근부담
# 내부통근 OD의 요금은 10번에서 의도적으로 결측이므로
# 11번 대표 편도교통비를 내부통근 요금 보조값으로 사용
HOME_COMMUTE_FILE = (
    PROCESSED_DIR
    / "commute_burden_by_home_dong.csv"
)

for label, path in [
    ("14_2 유형화", TYPOLOGY_FILE),
    ("10번 경로", ROUTE_FILE),
    ("11번 통근부담", HOME_COMMUTE_FILE),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} 파일이 없습니다: {path}"
        )

print("14_2 유형화:", TYPOLOGY_FILE)
print("10번 경로:", ROUTE_FILE)
print("11번 통근부담:", HOME_COMMUTE_FILE)


14_2 유형화: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/dong_typology_final.csv
10번 경로: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_routes_analysis_ready.csv
11번 통근부담: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_by_home_dong.csv


In [3]:
# =========================================================
# 3. 공통 함수
# =========================================================

def read_csv_clean(path):
    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        low_memory=False,
    )
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    return df


def normalize_admin_code_value(value):
    if pd.isna(value):
        return pd.NA

    text = re.sub(r"\.0$", "", str(value).strip())
    digits = re.sub(r"\D", "", text)

    if len(digits) == 8:
        return digits + "00"
    if len(digits) == 10:
        return digits

    return pd.NA


def normalize_code(series):
    return (
        series.map(normalize_admin_code_value)
        .astype("string")
    )


def find_col(df, candidates, required=True):
    col = next(
        (c for c in candidates if c in df.columns),
        None,
    )

    if col is None and required:
        raise KeyError(
            f"필요 컬럼을 찾지 못했습니다. 후보={candidates}\n"
            f"현재 컬럼={df.columns.tolist()}"
        )

    return col


def safe_spearman(x, y):
    mask = pd.Series(x).notna() & pd.Series(y).notna()

    if mask.sum() < 3:
        return np.nan, np.nan

    rho, p = spearmanr(
        pd.Series(x)[mask],
        pd.Series(y)[mask],
    )
    return rho, p


def minmax_good_low(series):
    s = pd.to_numeric(series, errors="coerce")
    lo = s.min()
    hi = s.max()

    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(50.0, index=s.index)

    return 100 * (hi - s) / (hi - lo)


def rank_within_group(df, group_col, value_col, ascending=True):
    return (
        df.groupby(group_col)[value_col]
        .rank(
            method="min",
            ascending=ascending,
        )
    )


def topk_overlap(a, b, k=10):
    sa = set(a[:k])
    sb = set(b[:k])
    return len(sa & sb) / k if k else np.nan

## 4. 경로 데이터 표준화

경로 파일 버전에 따라 컬럼명이 조금 다를 수 있으므로,
여러 후보 이름을 자동 탐색해 공통 스키마로 정리한다.

필수적으로 필요한 값:
- 거주동 코드/이름
- 근무동 코드/이름
- 편도 통근시간
- 대중교통 요금

거리와 환승 정보가 있으면 추가 보존한다.

In [4]:
# =========================================================
# 4. 10번 최종 경로 데이터 표준화
# =========================================================

routes_raw = read_csv_clean(ROUTE_FILE)
typology_raw = read_csv_clean(TYPOLOGY_FILE)
home_commute_raw = read_csv_clean(HOME_COMMUTE_FILE)

print("[10번 경로 원본]")
print(routes_raw.shape)
display(routes_raw.head())

print("[14_2 유형화 원본]")
print(typology_raw.shape)
display(typology_raw.head())

print("[11번 거주동별 통근부담 원본]")
print(home_commute_raw.shape)
display(home_commute_raw.head())

# 10번 노트북의 실제 최종 컬럼 기준
required_route_cols = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "내부통근여부",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
]

missing = [
    c for c in required_route_cols
    if c not in routes_raw.columns
]

if missing:
    raise KeyError(
        "10번 최종 CSV에 필요한 컬럼이 없습니다.\n"
        f"없는 컬럼: {missing}\n"
        f"현재 컬럼: {routes_raw.columns.tolist()}"
    )

routes = pd.DataFrame({
    "OD_KEY": routes_raw["OD_KEY"].astype("string"),
    "거주동코드": normalize_code(routes_raw["거주동 코드"]),
    "거주동명": routes_raw["거주동 이름"].astype("string"),
    "근무동코드": normalize_code(routes_raw["근무동 코드"]),
    "근무동명": routes_raw["근무동 이름"].astype("string"),
    "내부통근여부": routes_raw["내부통근여부"],
    "편도통근시간_분": pd.to_numeric(
        routes_raw["분석용_편도시간_분"],
        errors="coerce",
    ),
    "편도통근거리_km": pd.to_numeric(
        routes_raw["분석용_편도거리_km"],
        errors="coerce",
    ),
    "편도교통비_원": pd.to_numeric(
        routes_raw["분석용_편도요금_원"],
        errors="coerce",
    ),
})

# 분석에 유용한 원본 OD 지표 보존
for src_col, new_col in [
    ("출근_이동량", "출근_이동량"),
    ("최종_가중치", "최종_가중치"),
    ("목적지_출근비중", "목적지_출근비중"),
    ("누적_출근비중", "누적_출근비중"),
    ("환승횟수", "환승횟수"),
    ("총도보시간_분", "총도보시간_분"),
    ("도보시간비중", "도보시간비중"),
    ("요금산출방식", "요금산출방식"),
    ("경로값_산출방식", "경로값_산출방식"),
]:
    if src_col in routes_raw.columns:
        routes[new_col] = routes_raw[src_col]

# 내부통근여부 자료형 정리
if routes["내부통근여부"].dtype != bool:
    routes["내부통근여부"] = (
        routes["내부통근여부"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

internal_missing = routes["내부통근여부"].isna()

routes.loc[
    internal_missing,
    "내부통근여부",
] = (
    routes.loc[internal_missing, "거주동코드"]
    .eq(routes.loc[internal_missing, "근무동코드"])
)

routes["내부통근여부"] = (
    routes["내부통근여부"].astype(bool)
)

# 기본 품질검사
if routes["OD_KEY"].duplicated().any():
    raise ValueError("10번 경로 데이터에 중복 OD_KEY가 있습니다.")

if routes["편도통근시간_분"].isna().any():
    raise ValueError("편도 통근시간 결측이 있습니다.")

external_fee_missing = (
    (~routes["내부통근여부"])
    & routes["편도교통비_원"].isna()
).sum()

if external_fee_missing > 0:
    raise ValueError(
        "외부 통근 요금 결측이 남아 있습니다. "
        f"결측 OD 수: {external_fee_missing:,}"
    )

print("표준화 OD:", f"{len(routes):,}")
print("내부통근 OD:", f"{routes['내부통근여부'].sum():,}")
print(
    "내부통근 요금 결측:",
    f"{routes.loc[routes['내부통근여부'], '편도교통비_원'].isna().sum():,}",
)


[10번 경로 원본]
(30839, 35)


,OD_KEY,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,선택목적지_출근량합,최종_가중치,평균_이동시간_분,평균_이동거리_m,내부통근여부,분석용_편도시간_분,분석용_편도거리_km,분석용_편도요금_원,요금산출방식,총도보시간_분,총도보거리_m,도보시간비중,환승횟수,버스_이용구간수,지하철_이용구간수,도보_구간수,기차_이용구간수,교통수단_순서,이용노선,추천경로수,전체추천경로수,최종경로유형,경로값_산출방식,경로정보존재여부,요금정보존재여부
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,"55,710.2500","564,412.0400",0.0987,0.0987,"451,971.5100",0.1233,17.6800,928.2000,False,16.9000,1.4700,"1,500.0000",API_선택경로요금,14.8000,976.0000,0.8757,0.0000,1.0000,0.0000,2.0000,0.0000,WALK → BUS → WALK,지선:1711,4.0000,4.0000,대중교통,TMAP_최종경로,True,True
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,"47,373.5200","564,412.0400",0.0839,0.1826,"451,971.5100",0.1048,25.7200,"1,759.8300",False,18.6000,2.8200,"1,500.0000",API_선택경로요금,7.7000,557.0000,0.4140,1.0000,2.0000,0.0000,3.0000,0.0000,WALK → BUS → WALK → BUS → WALK,지선:1020 → 간선:272,10.0000,10.0000,대중교통,TMAP_최종경로,True,True
2,11110515_11110515,11110515,청운효자동,11110515,청운효자동,3,"43,228.9200","564,412.0400",0.0766,0.2592,"451,971.5100",0.0956,15.4600,502.6900,True,15.4600,0.5027,NaN,내부통근_요금미산출,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,내부통근_원본OD,원본_OD_내부통근,True,False
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,"21,721.1700","564,412.0400",0.0385,0.2977,"451,971.5100",0.0481,30.4600,"2,213.6200",False,20.1000,5.9500,"1,500.0000",API_선택경로요금,6.0000,402.0000,0.2985,1.0000,2.0000,0.0000,3.0000,0.0000,WALK → BUS → WALK → BUS → WALK,지선:7022 → 순환:TOUR11,10.0000,10.0000,대중교통,TMAP_최종경로,True,True
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,"17,247.3800","564,412.0400",0.0306,0.3283,"451,971.5100",0.0382,30.1900,"2,127.8800",False,12.7000,3.0000,"1,500.0000",API_선택경로요금,5.7000,313.0000,0.4488,0.0000,1.0000,0.0000,2.0000,0.0000,WALK → BUS → WALK,지선:1711,10.0000,10.0000,대중교통,TMAP_최종경로,True,True


[14_2 유형화 원본]
(427, 25)


,행정동코드,시군구명,행정동명,권역,군집,행정동_유형,4사분면_주거통근,4사분면_부담구조,자기군집중심거리,두번째군집중심거리,경계거리차이,군집경계_모호,군집내_중심거리백분위,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,표면주거비_대체여부,표면주거비_대체방법,표면주거비_참고동,청년1인세대_비율_보정여부,청년1인세대_비율_보정방법
0,1111051500,종로구,청운효자동,도심권,2,근접통근 균형형,고주거비·저통근부담,저부담·지역집중,0.1812,0.6845,0.5033,False,0.0506,"718,483.7093",26.2809,"64,666.0717",0.6325,0.7440,0.0311,13.2000,False,원천관측값,NaN,False,원천관측값
1,1111053000,종로구,사직동,도심권,6,고주거비·직주근접형,고주거비·저통근부담,고부담·지역집중,0.4948,1.5536,1.0588,False,0.1875,"943,454.2971",21.4364,"64,422.2487",0.7153,0.6164,0.0835,21.2700,False,원천관측값,NaN,False,원천관측값
2,1111054000,종로구,삼청동,도심권,2,근접통근 균형형,고주거비·저통근부담,저부담·지역집중,0.7952,0.9874,0.1922,False,0.8861,"726,250.0000",25.6785,"59,473.4193",0.6632,0.6823,0.0501,14.5400,False,원천관측값,NaN,False,원천관측값
3,1111055000,종로구,부암동,도심권,5,저주거비·지역연계형,고주거비·고통근부담,고부담·지역집중,0.2720,0.7092,0.4372,False,0.1026,"645,093.6971",32.5494,"64,874.5332",0.6474,0.7651,0.0245,13.6100,False,원천관측값,NaN,False,원천관측값
4,1111056000,종로구,평창동,도심권,5,저주거비·지역연계형,고주거비·고통근부담,고부담·광역분산,0.4446,0.7662,0.3217,False,0.4615,"713,569.7619",33.3110,"66,590.2081",0.6004,0.7776,0.0225,7.5700,False,원천관측값,NaN,False,원천관측값


[11번 거주동별 통근부담 원본]
(428, 22)


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km,대표_편도교통비_원,월_통근시간_분,월_통근시간_시간,월_통근교통비_원,월_통근시간_기회비용_원,주요_출근목적지_목록,주요목적지_누적출근비중,거주동_전체_출근량,선택목적지_출근량합,API_경로포함률,교통비_산출포함률,내부통근비중,대표_편도도보시간_분,도보시간_산출포함률,대표_편도환승횟수,환승횟수_산출포함률,월평균_출근일수_가정,시간가치_원_시간
0,11110515,청운효자동,26.2809,6.0484,"1,539.6684","1,103.7980",18.3966,"64,666.0717","189,853.2641","사직동, 종로1.2.3.4가동, 청운효자동, 명동, 소공동",0.8008,"564,412.0400","451,971.5100",1.0000,0.9044,0.0956,9.0165,0.9044,0.8330,0.9044,21.0000,"10,320.0000"
1,11110530,사직동,21.4364,3.6259,"1,533.8631",900.3289,15.0055,"64,422.2487","154,856.5719","사직동, 종로1.2.3.4가동, 명동, 여의동, 소공동",0.8009,"636,380.7500","509,677.1300",1.0000,0.7039,0.2961,12.0132,0.7039,0.4278,0.7039,21.0000,"10,320.0000"
2,11110540,삼청동,25.6785,5.1693,"1,416.0338","1,078.4990",17.9750,"59,473.4193","185,501.8224","종로1.2.3.4가동, 삼청동, 청운효자동, 사직동, 가회동",0.8011,"106,609.7700","85,406.6800",1.0000,0.8878,0.1122,9.4489,0.8878,0.7510,0.8878,21.0000,"10,320.0000"
3,11110550,부암동,32.5494,7.5753,"1,544.6317","1,367.0754",22.7846,"64,874.5332","235,136.9694","종로1.2.3.4가동, 부암동, 사직동, 명동, 평창동",0.8020,"452,322.2600","362,753.5700",1.0000,0.9234,0.0766,10.2310,0.9234,0.8883,0.9234,21.0000,"10,320.0000"
4,11110560,평창동,33.3110,9.4207,"1,585.4811","1,399.0636",23.3177,"66,590.2081","240,638.9374","종로1.2.3.4가동, 평창동, 사직동, 부암동, 명동",0.8006,"703,357.4500","563,113.5100",1.0000,0.9125,0.0875,7.4039,0.9125,0.9922,0.9125,21.0000,"10,320.0000"


표준화 OD: 30,839
내부통근 OD: 428
내부통근 요금 결측: 428


## 5. 14_2 유형화 결과 결합

`dong_typology_final.csv`에서 거주동의:
- 표면주거비
- 6개 유형
- 4사분면
- 청년 1인세대 비율
- 권역/통근구조 지표

를 가져온다.

In [5]:
# =========================================================
# 5. 14_2 유형화 + 11번 대표 교통비 결합
# =========================================================

# ---------------------------------------------------------
# 1) 14_2 유형화
# ---------------------------------------------------------
typology = typology_raw.copy()

if "행정동코드" not in typology.columns:
    raise KeyError(
        "dong_typology_final.csv에 '행정동코드'가 없습니다.\n"
        f"현재 컬럼: {typology.columns.tolist()}"
    )

typology["거주동코드"] = normalize_code(
    typology["행정동코드"]
)

TYPOLOGY_KEEP_CANDIDATES = [
    "거주동코드",
    "시군구명",
    "행정동명",
    "권역",
    "군집",
    "행정동_유형",
    "4사분면_주거통근",
    "4사분면_부담구조",
    "표면주거비_원",
    "대표_편도통근시간_분",
    "월통근교통비_원",
    "동일통근권_내부출근비율",
    "목적지_정규화엔트로피",
    "목적지_HHI",
    "청년1인세대_비율",
]

typology_keep = [
    c for c in TYPOLOGY_KEEP_CANDIDATES
    if c in typology.columns
]

typology = (
    typology[typology_keep]
    .drop_duplicates("거주동코드")
)

# ---------------------------------------------------------
# 2) 11번 거주동별 대표 편도교통비
# ---------------------------------------------------------
required_home_cols = [
    "거주동 코드",
    "대표_편도교통비_원",
]

missing_home = [
    c for c in required_home_cols
    if c not in home_commute_raw.columns
]

if missing_home:
    raise KeyError(
        "11번 결과에 필요한 컬럼이 없습니다.\n"
        f"없는 컬럼: {missing_home}\n"
        f"현재 컬럼: {home_commute_raw.columns.tolist()}"
    )

home_fee = home_commute_raw[
    required_home_cols
].copy()

home_fee["거주동코드"] = normalize_code(
    home_fee["거주동 코드"]
)

home_fee["거주동대표_편도교통비_원"] = pd.to_numeric(
    home_fee["대표_편도교통비_원"],
    errors="coerce",
)

home_fee = (
    home_fee[
        [
            "거주동코드",
            "거주동대표_편도교통비_원",
        ]
    ]
    .drop_duplicates("거주동코드")
)

# ---------------------------------------------------------
# 3) 결합
# ---------------------------------------------------------
analysis = (
    routes
    .merge(
        typology,
        on="거주동코드",
        how="left",
        validate="many_to_one",
        suffixes=("", "_유형화"),
    )
    .merge(
        home_fee,
        on="거주동코드",
        how="left",
        validate="many_to_one",
    )
)

# 경로 파일의 거주동명을 기본 이름으로 사용
if "행정동명" in analysis.columns:
    analysis["행정동명"] = (
        analysis["행정동명"]
        .fillna(analysis["거주동명"])
    )
else:
    analysis["행정동명"] = analysis["거주동명"]

# ---------------------------------------------------------
# 4) 내부통근 요금 보조
# ---------------------------------------------------------
# 10번/11번 원칙:
# 내부통근 요금은 0원으로 두지 않는다.
# 15번은 OD별 총부담 계산이 필요하므로,
# 내부통근 OD에 한해서 11번의 거주동 대표 편도교통비를 proxy로 사용한다.
analysis["편도교통비_원_원본"] = (
    analysis["편도교통비_원"]
)

analysis["교통비_15번산출방식"] = np.where(
    analysis["편도교통비_원"].notna(),
    "10번_OD경로요금",
    pd.NA,
)

internal_fee_mask = (
    analysis["내부통근여부"]
    & analysis["편도교통비_원"].isna()
)

analysis.loc[
    internal_fee_mask,
    "편도교통비_원",
] = analysis.loc[
    internal_fee_mask,
    "거주동대표_편도교통비_원",
]

analysis.loc[
    internal_fee_mask,
    "교통비_15번산출방식",
] = "내부통근_11번거주동대표요금_proxy"

remaining_fee_missing = (
    analysis["편도교통비_원"].isna().sum()
)

print("경로 행:", f"{len(routes):,}")
print("유형화 결합 후:", f"{len(analysis):,}")
print(
    "표면주거비 결측:",
    f"{analysis['표면주거비_원'].isna().sum():,}"
)
print(
    "유형 결측:",
    f"{analysis['행정동_유형'].isna().sum():,}"
    if "행정동_유형" in analysis.columns
    else "행정동_유형 컬럼 없음"
)
print(
    "15번 분석용 교통비 최종 결측:",
    f"{remaining_fee_missing:,}"
)

if remaining_fee_missing > 0:
    print(
        "주의: 교통비가 끝까지 결측인 OD는 "
        "총부담 분석에서 제외됩니다."
    )

display(
    analysis[
        [
            "OD_KEY",
            "거주동코드",
            "거주동명",
            "근무동코드",
            "근무동명",
            "내부통근여부",
            "편도통근시간_분",
            "편도교통비_원_원본",
            "편도교통비_원",
            "교통비_15번산출방식",
            "표면주거비_원",
            "행정동_유형",
        ]
    ].head(20)
)


경로 행: 30,839
유형화 결합 후: 30,839
표면주거비 결측: 77
유형 결측: 77
15번 분석용 교통비 최종 결측: 0


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도교통비_원_원본,편도교통비_원,교통비_15번산출방식,표면주거비_원,행정동_유형
0,11110515_11110530,1111051500,청운효자동,1111053000,사직동,False,16.9000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
1,11110515_11110615,1111051500,청운효자동,1111061500,종로1.2.3.4가동,False,18.6000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
2,11110515_11110515,1111051500,청운효자동,1111051500,청운효자동,True,15.4600,NaN,"1,539.6684",내부통근_11번거주동대표요금_proxy,"718,483.7093",근접통근 균형형
3,11110515_11140550,1111051500,청운효자동,1114055000,명동,False,20.1000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
4,11110515_11140520,1111051500,청운효자동,1114052000,소공동,False,12.7000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
5,11110515_11560540,1111051500,청운효자동,1156054000,여의동,False,40.4000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
6,11110515_11680640,1111051500,청운효자동,1168064000,역삼1동,False,51.8000,"1,600.0000","1,600.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
7,11110515_11140540,1111051500,청운효자동,1114054000,회현동,False,22.8000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
8,11110515_11110600,1111051500,청운효자동,1111060000,가회동,False,17.6000,"1,400.0000","1,400.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형
9,11110515_11110640,1111051500,청운효자동,1111064000,이화동,False,22.4000,"1,500.0000","1,500.0000",10번_OD경로요금,"718,483.7093",근접통근 균형형


## 6. 공통 근무동–거주동 분석 테이블

메인 분석의 시간비용은 특정 월소득을 임의로 가정하지 않고,
**공통 시간가치 10,320원/시간**을 적용한다.

- 월 통근시간 = 편도 통근시간 × 2 × 월 출근일수
- 월 교통비 = 편도 요금 × 2 × 월 출근일수
- 월 통근시간비용 = 월 통근시간 × 공통 시간가치
- 총 주거·통근 부담 = 표면주거비 + 월 교통비 + 월 통근시간비용

소득에 따른 차이는 메인 결과와 분리하여 뒤의 **소득 민감도 분석**에서만 계산한다.


In [6]:
# =========================================================
# 6. 공통 부담 계산
# =========================================================

def add_burden_metrics(
    df,
    work_days=WORK_DAYS_PER_MONTH,
    time_value_per_hour=None,
    monthly_income=None,
    time_value_factor=1.0,
):
    out = df.copy()

    # 1) 메인 분석: 시간당 공통가치 직접 지정
    if time_value_per_hour is not None:
        commute_hour_value = (
            float(time_value_per_hour)
            * float(time_value_factor)
        )
        value_basis = "공통시간가치"

    # 2) 소득 민감도: 월소득 -> 시간당 임금 환산
    elif monthly_income is not None:
        hourly_wage = (
            float(monthly_income)
            / MONTHLY_WORK_HOURS
        )
        commute_hour_value = (
            hourly_wage
            * float(time_value_factor)
        )
        value_basis = "월소득환산"

    else:
        raise ValueError(
            "time_value_per_hour 또는 monthly_income 중 "
            "하나는 지정해야 합니다."
        )

    out["월통근시간_시간"] = (
        out["편도통근시간_분"]
        * 2
        * work_days
        / 60
    )

    out["월교통비_원"] = (
        out["편도교통비_원"]
        * 2
        * work_days
    )

    out["월통근시간비용_원"] = (
        out["월통근시간_시간"]
        * commute_hour_value
    )

    out["총주거통근부담_원"] = (
        out["표면주거비_원"]
        + out["월교통비_원"]
        + out["월통근시간비용_원"]
    )

    out["주거비비중"] = (
        out["표면주거비_원"]
        / out["총주거통근부담_원"]
    )

    out["통근비용비중"] = (
        (
            out["월교통비_원"]
            + out["월통근시간비용_원"]
        )
        / out["총주거통근부담_원"]
    )

    out["시간가치기준"] = value_basis
    out["통근시간가치_시간당원"] = commute_hour_value

    if monthly_income is not None:
        out["월소득_원"] = monthly_income
    else:
        out["월소득_원"] = pd.NA

    out["시간가치계수"] = time_value_factor

    return out


# 메인 분석은 특정 월소득이 아니라 공통 시간가치 기준
common = add_burden_metrics(
    analysis,
    time_value_per_hour=BASE_TIME_VALUE_PER_HOUR,
    time_value_factor=BASE_TIME_VALUE_FACTOR,
)

common = common[
    common["편도통근시간_분"].between(
        1,
        MAX_ONEWAY_MINUTES,
    )
].dropna(
    subset=[
        "표면주거비_원",
        "편도통근시간_분",
        "편도교통비_원",
        "총주거통근부담_원",
    ]
).copy()

COMMON_COLUMNS = [
    "OD_KEY",
    "근무동코드",
    "근무동명",
    "거주동코드",
    "거주동명",
    "행정동_유형",
    "4사분면_주거통근",
    "표면주거비_원",
    "편도통근시간_분",
    "월교통비_원",
    "월통근시간비용_원",
    "총주거통근부담_원",
    "시간가치기준",
    "통근시간가치_시간당원",
    "교통비_15번산출방식",
]

display(
    common[
        [c for c in COMMON_COLUMNS if c in common.columns]
    ].head(20)
)

print("최종 OD 조합:", f"{len(common):,}")
print("근무동 수:", common["근무동코드"].nunique())
print("거주동 수:", common["거주동코드"].nunique())
print(
    "내부통근 proxy 요금 사용 OD:",
    f"{(common['교통비_15번산출방식'] == '내부통근_11번거주동대표요금_proxy').sum():,}"
)


,OD_KEY,근무동코드,근무동명,거주동코드,거주동명,행정동_유형,4사분면_주거통근,표면주거비_원,편도통근시간_분,월교통비_원,월통근시간비용_원,총주거통근부담_원,시간가치기준,통근시간가치_시간당원,교통비_15번산출방식
0,11110515_11110530,1111053000,사직동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",16.9000,"63,000.0000","122,085.6000","903,569.3093",공통시간가치,"10,320.0000",10번_OD경로요금
1,11110515_11110615,1111061500,종로1.2.3.4가동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",18.6000,"63,000.0000","134,366.4000","915,850.1093",공통시간가치,"10,320.0000",10번_OD경로요금
2,11110515_11110515,1111051500,청운효자동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",15.4600,"64,666.0717","111,683.0400","894,832.8210",공통시간가치,"10,320.0000",내부통근_11번거주동대표요금_proxy
3,11110515_11140550,1114055000,명동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",20.1000,"63,000.0000","145,202.4000","926,686.1093",공통시간가치,"10,320.0000",10번_OD경로요금
4,11110515_11140520,1114052000,소공동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",12.7000,"63,000.0000","91,744.8000","873,228.5093",공통시간가치,"10,320.0000",10번_OD경로요금
5,11110515_11560540,1156054000,여의동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",40.4000,"63,000.0000","291,849.6000","1,073,333.3093",공통시간가치,"10,320.0000",10번_OD경로요금
6,11110515_11680640,1168064000,역삼1동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",51.8000,"67,200.0000","374,203.2000","1,159,886.9093",공통시간가치,"10,320.0000",10번_OD경로요금
7,11110515_11140540,1114054000,회현동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",22.8000,"63,000.0000","164,707.2000","946,190.9093",공통시간가치,"10,320.0000",10번_OD경로요금
8,11110515_11110600,1111060000,가회동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",17.6000,"58,800.0000","127,142.4000","904,426.1093",공통시간가치,"10,320.0000",10번_OD경로요금
9,11110515_11110640,1111064000,이화동,1111051500,청운효자동,근접통근 균형형,고주거비·저통근부담,"718,483.7093",22.4000,"63,000.0000","161,817.6000","943,301.3093",공통시간가치,"10,320.0000",10번_OD경로요금


최종 OD 조합: 30,732
근무동 수: 428
거주동 수: 427
내부통근 proxy 요금 사용 OD: 427


# 분석 1. 주거비를 줄이면 통근 부담이 얼마나 늘어나는가

근무동별로 따로 비교한다.

기준점은 해당 근무동까지 **30분 이내 거주동들의 중위 표면주거비와 가장 가까운 실제 거주동**으로 둔다.

주의할 점은 `10만 원 절감당` 비율이다. 실행 결과에서 기준보다 몇 천 원만 저렴한 동까지
100,000원을 곱해 환산하면서 수천 분 같은 비정상적으로 큰 평균이 나타났다.

따라서:
- 실질 순절감액/월세 착시 판정은 모든 저렴한 후보에서 계산
- **10만 원 절감당 지표는 실제로 10만 원 이상 절감되는 후보에서만 계산**
- 평균보다 중앙값을 핵심값으로 우선 해석

하도록 수정한다.


In [7]:
# =========================================================
# 7. 분석 1 — 근무동별 기준값
# =========================================================

baseline_rows = []

for work_code, g in common.groupby("근무동코드"):
    near = g[
        g["편도통근시간_분"] <= BASELINE_NEARBY_MINUTES
    ].copy()

    if len(near) == 0:
        continue

    # 30분 이내 거주 후보의 중위 주거비
    baseline_housing = near["표면주거비_원"].median()

    # 중위 주거비와 가장 가까운 실제 동을 기준 대표동으로 선택
    ref_idx = (
        near["표면주거비_원"] - baseline_housing
    ).abs().idxmin()

    ref = near.loc[ref_idx]

    baseline_rows.append({
        "근무동코드": work_code,
        "기준거주동코드": ref["거주동코드"],
        "기준거주동명": ref.get("행정동명", pd.NA),
        "기준표면주거비_원": ref["표면주거비_원"],
        "기준편도통근시간_분": ref["편도통근시간_분"],
        "기준월교통비_원": ref["월교통비_원"],
        "기준월통근시간비용_원": ref["월통근시간비용_원"],
        "기준총부담_원": ref["총주거통근부담_원"],
        "30분이내후보수": len(near),
    })

baseline = pd.DataFrame(baseline_rows)

print("기준값 생성 근무동 수:", len(baseline))
display(baseline.head(20))

기준값 생성 근무동 수: 428


,근무동코드,기준거주동코드,기준거주동명,기준표면주거비_원,기준편도통근시간_분,기준월교통비_원,기준월통근시간비용_원,기준총부담_원,30분이내후보수
0,1111051500,1141056500,충현동,"693,214.3742",26.1000,"63,000.0000","188,546.4000","944,760.7742",31
1,1111053000,1141056500,충현동,"693,214.3742",19.6000,"63,000.0000","141,590.4000","897,804.7742",100
2,1111054000,1129052500,성북동,"713,521.8729",22.6000,"63,000.0000","163,262.4000","939,784.2729",15
3,1111055000,1111055000,부암동,"645,093.6971",16.4300,"64,874.5332","118,690.3200","828,658.5503",10
4,1111056000,1129063000,정릉제2동,"643,491.2212",14.7000,"63,000.0000","106,192.8000","812,684.0212",11
5,1111057000,1141062000,홍제제1동,"657,729.7762",11.1000,"65,100.0000","80,186.4000","803,016.1762",6
6,1111058000,1111055000,부암동,"645,093.6971",21.8000,"63,000.0000","157,483.2000","865,576.8971",40
7,1111060000,1129055500,삼선동,"661,862.5222",16.5000,"63,000.0000","119,196.0000","844,058.5222",65
8,1111061500,1141056500,충현동,"693,214.3742",23.3000,"65,100.0000","168,319.2000","926,633.5742",98
9,1111063000,1120079000,용답동,"667,895.5328",19.8000,"63,000.0000","143,035.2000","873,930.7328",100


In [8]:
# =========================================================
# 8. 분석 1 — 주거비 절감 vs 통근부담 증가
# =========================================================

tradeoff = common.merge(
    baseline,
    on="근무동코드",
    how="inner",
)

tradeoff["주거비절감액_원"] = (
    tradeoff["기준표면주거비_원"]
    - tradeoff["표면주거비_원"]
)

tradeoff["편도통근시간증가_분"] = (
    tradeoff["편도통근시간_분"]
    - tradeoff["기준편도통근시간_분"]
)

tradeoff["월교통비증가_원"] = (
    tradeoff["월교통비_원"]
    - tradeoff["기준월교통비_원"]
)

tradeoff["월통근시간비용증가_원"] = (
    tradeoff["월통근시간비용_원"]
    - tradeoff["기준월통근시간비용_원"]
)

tradeoff["통근부담증가액_원"] = (
    tradeoff["월교통비증가_원"]
    + tradeoff["월통근시간비용증가_원"]
)

tradeoff["실질순절감액_원"] = (
    tradeoff["주거비절감액_원"]
    - tradeoff["통근부담증가액_원"]
)

tradeoff["손익분기주거비절감액_원"] = (
    tradeoff["통근부담증가액_원"]
)

# 기준보다 저렴한 후보 전체
cheaper_all = tradeoff[
    tradeoff["주거비절감액_원"] > 0
].copy()

cheaper_all["절감판정"] = np.where(
    cheaper_all["실질순절감액_원"] > 0,
    "실제 부담 절감",
    "월세 착시",
)

# '10만원 절감당' 비율 분석은 실제 절감액 >= 10만원인 후보만 사용
cheaper = cheaper_all[
    cheaper_all["주거비절감액_원"]
    >= MIN_SAVING_FOR_100K_METRIC
].copy()

scale = (
    100_000
    / cheaper["주거비절감액_원"]
)

cheaper["10만원절감당_통근시간증가_분"] = (
    cheaper["편도통근시간증가_분"]
    * scale
)

cheaper["10만원절감당_월교통비증가_원"] = (
    cheaper["월교통비증가_원"]
    * scale
)

cheaper["10만원절감당_시간비용증가_원"] = (
    cheaper["월통근시간비용증가_원"]
    * scale
)

tradeoff_summary = (
    cheaper.groupby("근무동코드")
    .agg(
        분석후보수=("거주동코드", "count"),
        중앙값_10만원절감당_통근시간증가_분=(
            "10만원절감당_통근시간증가_분",
            "median",
        ),
        평균_10만원절감당_통근시간증가_분=(
            "10만원절감당_통근시간증가_분",
            "mean",
        ),
        중앙값_10만원절감당_월교통비증가_원=(
            "10만원절감당_월교통비증가_원",
            "median",
        ),
        중앙값_10만원절감당_시간비용증가_원=(
            "10만원절감당_시간비용증가_원",
            "median",
        ),
        중앙값_실질순절감액_원=(
            "실질순절감액_원",
            "median",
        ),
        월세착시비율=(
            "절감판정",
            lambda s: (s == "월세 착시").mean(),
        ),
    )
    .reset_index()
)

print(
    "10만원 절감당 비율 계산 대상:",
    f"{len(cheaper):,}개 "
    f"(실제 절감액 {MIN_SAVING_FOR_100K_METRIC:,}원 이상)"
)

display(
    tradeoff_summary.sort_values(
        "중앙값_10만원절감당_통근시간증가_분",
        ascending=False,
    ).head(30)
)


10만원 절감당 비율 계산 대상: 8,508개 (실제 절감액 100,000원 이상)


,근무동코드,분석후보수,중앙값_10만원절감당_통근시간증가_분,평균_10만원절감당_통근시간증가_분,중앙값_10만원절감당_월교통비증가_원,중앙값_10만원절감당_시간비용증가_원,중앙값_실질순절감액_원,월세착시비율
257,1156054000,81,28.3143,27.4097,"4,503.0252","204,542.6295","-161,292.6145",0.7778
167,1138057000,4,27.3556,25.8104,"3,540.4888","197,617.1388","-139,961.8342",0.7500
83,1123060000,9,26.9557,28.1002,0.0000,"194,727.7373","-120,960.1510",1.0000
239,1153054000,20,25.2557,19.6420,"8,147.2882","182,447.3010","-113,246.6074",0.5500
59,1120066000,63,24.6443,25.2483,"3,563.2046","178,030.2205","-117,556.8877",0.9365
61,1120069000,111,24.1881,26.0258,"4,253.0959","174,734.6200","-123,453.0432",0.9459
96,1126061000,1,24.0779,24.0779,"5,745.8737","173,939.0897","-87,369.7009",1.0000
82,1123057000,1,23.8177,23.8177,0.0000,"172,058.8268","-75,030.8122",1.0000
146,1135057000,2,23.5266,23.5266,"-12,015.8807","169,956.2240","-70,552.1501",1.0000
227,1150060300,49,21.7612,23.5890,"5,394.1844","157,203.1669","-93,443.3861",0.8367


### 분석 1 보조분석 — 산점도, 회귀, 10만 원 주거비 구간

문서의 권고대로 인과관계가 아니라 **관계/경향**으로 해석한다.

회귀식은 근무동별 단순 관계를 보기 위한 보조자료이고,
사용자 설명에서는 10만 원 주거비 구간별 평균 통근시간이 더 직관적이다.

In [9]:
# =========================================================
# 9. 분석 1 — 근무동별 주거비/통근시간 관계
# =========================================================

regression_rows = []
housing_band_rows = []

for work_code, g in common.groupby("근무동코드"):
    if len(g) < 10:
        continue

    x = g[["표면주거비_원"]].to_numpy()
    y = g["편도통근시간_분"].to_numpy()

    model = LinearRegression()
    model.fit(x, y)

    # 주거비가 10만원 낮아질 때의 시간 변화
    effect_per_minus_100k = (
        model.coef_[0] * -100_000
    )

    rho, p = safe_spearman(
        g["표면주거비_원"],
        g["편도통근시간_분"],
    )

    regression_rows.append({
        "근무동코드": work_code,
        "후보수": len(g),
        "주거비10만원감소_예상통근시간변화_분": effect_per_minus_100k,
        "회귀_R2": model.score(x, y),
        "Spearman_rho": rho,
        "Spearman_p": p,
    })

    temp = g.copy()
    temp["주거비_10만원구간"] = (
        np.floor(temp["표면주거비_원"] / 100_000)
        * 100_000
    )

    band = (
        temp.groupby("주거비_10만원구간")
        .agg(
            행정동수=("거주동코드", "count"),
            평균편도통근시간_분=("편도통근시간_분", "mean"),
            평균월교통비_원=("월교통비_원", "mean"),
        )
        .reset_index()
    )

    band["근무동코드"] = work_code
    housing_band_rows.append(band)

tradeoff_regression = pd.DataFrame(regression_rows)

housing_band_summary = (
    pd.concat(
        housing_band_rows,
        ignore_index=True,
    )
    if housing_band_rows
    else pd.DataFrame()
)

display(tradeoff_regression.head(30))
display(housing_band_summary.head(30))

,근무동코드,후보수,주거비10만원감소_예상통근시간변화_분,회귀_R2,Spearman_rho,Spearman_p
0,1111051500,49,2.7063,0.1104,-0.4859,0.0004
1,1111053000,408,1.6700,0.0321,-0.2524,0.0000
2,1111054000,22,1.1244,0.0124,-0.3327,0.1303
3,1111055000,12,3.0917,0.1349,-0.5315,0.0754
4,1111056000,15,-1.7809,0.0558,-0.2286,0.4126
5,1111058000,50,1.0965,0.0380,-0.4952,0.0003
6,1111060000,155,0.3909,0.0024,-0.1519,0.0591
7,1111061500,426,1.7353,0.0452,-0.2636,0.0000
8,1111063000,262,2.0901,0.0506,-0.3644,0.0000
9,1111064000,203,1.1164,0.0154,-0.1988,0.0045


,주거비_10만원구간,행정동수,평균편도통근시간_분,평균월교통비_원,근무동코드
0,"400,000.0000",4,35.2750,"64,050.0000",1111051500
1,"500,000.0000",10,34.5500,"63,630.0000",1111051500
2,"600,000.0000",17,25.8176,"63,741.1765",1111051500
3,"700,000.0000",13,23.9508,"63,289.6978",1111051500
4,"800,000.0000",3,32.1667,"67,200.0000",1111051500
5,"900,000.0000",2,19.2500,"60,900.0000",1111051500
6,"400,000.0000",30,46.7333,"68,670.0000",1111053000
7,"500,000.0000",113,43.9044,"66,884.0708",1111053000
8,"600,000.0000",133,39.4459,"66,363.1579",1111053000
9,"700,000.0000",80,36.2550,"66,806.2500",1111053000


# 분석 2. 어떤 지역이 실제로 비용 효율적인가

주거비만 싼 곳과 **총 주거·통근 부담이 낮은 곳**이 같은지 비교한다.

근무동별 중위값을 기준으로:
- A: 저주거비·저총부담 → 실질 비용 효율
- B: 저주거비·고총부담 → 월세 착시
- C: 고주거비·저총부담 → 숨은 효율
- D: 고주거비·고총부담 → 종합 부담 큼

을 분류한다.

In [10]:
# =========================================================
# 10. 분석 2 — 비용 효율 유형 / 효율지수
# =========================================================

efficiency = common.copy()

group_medians = (
    efficiency.groupby("근무동코드")
    .agg(
        근무동_중위주거비_원=("표면주거비_원", "median"),
        근무동_중위총부담_원=("총주거통근부담_원", "median"),
        근무동_최소총부담_원=("총주거통근부담_원", "min"),
    )
    .reset_index()
)

efficiency = efficiency.merge(
    group_medians,
    on="근무동코드",
    how="left",
)

low_housing = (
    efficiency["표면주거비_원"]
    <= efficiency["근무동_중위주거비_원"]
)
low_total = (
    efficiency["총주거통근부담_원"]
    <= efficiency["근무동_중위총부담_원"]
)

efficiency["비용효율유형"] = np.select(
    [
        low_housing & low_total,
        low_housing & ~low_total,
        ~low_housing & low_total,
        ~low_housing & ~low_total,
    ],
    [
        "A_실질비용효율",
        "B_월세착시",
        "C_숨은효율",
        "D_종합고부담",
    ],
)

efficiency["비용효율성지수"] = (
    efficiency["근무동_최소총부담_원"]
    / efficiency["총주거통근부담_원"]
    * 100
)

efficiency["주거비순위"] = rank_within_group(
    efficiency,
    "근무동코드",
    "표면주거비_원",
    ascending=True,
)

efficiency["총부담순위"] = rank_within_group(
    efficiency,
    "근무동코드",
    "총주거통근부담_원",
    ascending=True,
)

# 양수면 총부담 기준으로 순위 상승
efficiency["순위변화"] = (
    efficiency["주거비순위"]
    - efficiency["총부담순위"]
)

print("[비용 효율 유형]")
display(
    efficiency["비용효율유형"]
    .value_counts()
    .to_frame("OD수")
)

print("\n[월세 착시 예시]")
display(
    efficiency[
        efficiency["비용효율유형"] == "B_월세착시"
    ]
    .sort_values(
        "순위변화",
        ascending=True,
    )
    .head(30)
)

print("\n[숨은 효율 예시]")
display(
    efficiency[
        efficiency["비용효율유형"] == "C_숨은효율"
    ]
    .sort_values(
        "순위변화",
        ascending=False,
    )
    .head(30)
)

[비용 효율 유형]


,OD수
비용효율유형,
A_실질비용효율,11291
D_종합고부담,11051
B_월세착시,4198
C_숨은효율,4192



[월세 착시 예시]


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,4사분면_주거통근,4사분면_부담구조,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,월소득_원,시간가치계수,근무동_중위주거비_원,근무동_중위총부담_원,근무동_최소총부담_원,비용효율유형,비용효율성지수,주거비순위,총부담순위,순위변화
12097,11320680_11650520,1132068000,쌍문3동,1165052000,서초2동,False,89.3000,23.0300,"1,800.0000","2,640.7500",0.0044,0.0035,0.7462,1.0000,6.1000,0.0683,API_선택경로요금,TMAP_최종경로,도봉구,쌍문제3동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,고부담·광역분산,"508,011.3086",39.6987,"65,840.1676",0.4446,0.8422,0.0106,17.9600,"1,567.6230","1,800.0000",10번_OD경로요금,62.5100,"75,600.0000","645,103.2000","1,228,714.5086",0.4134,0.5866,공통시간가치,"10,320.0000",<NA>,1.0000,"643,601.9179","1,069,819.4675","797,133.3682",B_월세착시,64.8754,31.0000,343.0000,-312.0000
9827,11290810_11560540,1129081000,석관동,1156054000,여의동,False,88.0000,21.7000,"1,800.0000","30,009.7600",0.0231,0.0185,0.1942,1.0000,5.2000,0.0591,API_선택경로요금,TMAP_최종경로,성북구,석관동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,고부담·광역분산,"531,034.1305",36.0387,"70,029.7374",0.3655,0.8252,0.0165,15.8100,"1,667.3747","1,800.0000",10번_OD경로요금,61.6000,"75,600.0000","635,712.0000","1,242,346.1305",0.4274,0.5726,공통시간가치,"10,320.0000",<NA>,1.0000,"641,567.0476","1,069,541.2000","647,862.4641",B_월세착시,52.1483,60.0000,368.0000,-308.0000
10587,11305608_11680640,1130560800,번3동,1168064000,역삼1동,False,85.7000,21.2200,"1,500.0000","9,039.2500",0.0168,0.0135,0.2313,0.0000,11.6000,0.1354,API_선택경로요금,TMAP_최종경로,강북구,번3동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,저부담·광역분산,"499,444.7674",33.5991,"64,263.5754",0.4712,0.8565,0.0119,4.4100,"1,530.0851","1,500.0000",10번_OD경로요금,59.9900,"63,000.0000","619,096.8000","1,181,541.5674",0.4227,0.5773,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",B_월세착시,68.7894,33.0000,336.0000,-303.0000
10499,11305603_11680640,1130560300,번2동,1168064000,역삼1동,False,86.0000,22.2000,"1,500.0000","8,712.6100",0.0154,0.0123,0.3432,0.0000,9.5000,0.1105,API_선택경로요금,TMAP_최종경로,강북구,번2동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,저부담·광역분산,"487,188.0454",31.9879,"65,802.1582",0.4991,0.8264,0.0131,9.4200,"1,566.7181","1,500.0000",10번_OD경로요금,60.2000,"63,000.0000","621,264.0000","1,171,452.0454",0.4159,0.5841,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",B_월세착시,69.3818,27.0000,326.0000,-299.0000
13768,11350640_11680650,1135064000,상계2동,1168065000,역삼2동,False,86.2000,22.1600,"1,800.0000","5,798.8000",0.0079,0.0063,0.5990,1.0000,6.2000,0.0719,API_선택경로요금,TMAP_최종경로,노원구,상계2동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,고부담·광역분산,"528,835.5091",34.7327,"66,483.6777",0.4857,0.8165,0.0175,12.1500,"1,582.9447","1,800.0000",10번_OD경로요금,60.3400,"75,600.0000","622,708.8000","1,227,144.3091",0.4309,0.5691,공통시간가치,"10,320.0000",<NA>,1.0000,"643,948.5320","1,068,783.6660","815,689.0543",B_월세착시,66.4705,53.0000,348.0000,-295.0000
12140,11320681_11650530,1132068100,쌍문4동,1165053000,서초3동,False,84.2000,25.1500,"1,800.0000","8,117.5400",0.0126,0.0101,0.4738,2.0000,5.5000,0.0653,API_선택경로요금,TMAP_최종경로,도봉구,쌍문제4동,동북권,4.0000,주거절감·장거리통근형,저주거비·고통근부담,고부담·광역분산,"428,379.0850",38.7153,"66,971.6477",0.4732,0.8218,0.0133,5.6600,"1,594.5630","1,800.0000",10번_OD경로요금,58.9400,"75,600.0000","608,260.8000","1,112,239.8850",0.3851,0.6149,공통시간가치,"10,320.0000",<NA>,1.0000,"639,956.3195","1,050,574.5745","722,860.5682",B_월세착시,64.9914,3.0000,297.0000,-294.0000
18224,11470580_11680650,1147058000,신월3동,1168065000,역삼2동,False,87.0000,23.7800,"1,800.0000","2,185.8500",0.0065,0.0052,0.7338,1.0000,5.2000,0.0598,API_선택경로요금,TMAP_최종경로,양천구,신월3동,서남권,5.0000,저주거비·지역연계형,저주거비·저통근부담,저부담·지역집중,"453,126.4665",30.1502,"65,840.0492",0.7044,0.7876,0.0174,11.2800,"1,567.6202","1,800.0000",10번_OD경로요금,60.9000,"75,600.0000","628,488.0000","1,157,214.4665",0.3916,0.6084,공통시간가치,"10,320.0000",<NA>,1.0000,"643,948.5320","1,068,783.6660","815,689.0543",B_월세착시,70.4873,5.0000,299.0000,-294


[숨은 효율 예시]


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,4사분면_주거통근,4사분면_부담구조,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,월소득_원,시간가치계수,근무동_중위주거비_원,근무동_중위총부담_원,근무동_최소총부담_원,비용효율유형,비용효율성지수,주거비순위,총부담순위,순위변화
25498,11650510_11680640,1165051000,서초1동,1168064000,역삼1동,False,12.1000,1.5000,"1,200.0000","79,302.4400",0.0841,0.0675,0.3568,0.0000,7.3000,0.6033,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,고주거비·저통근부담,저부담·지역집중,"779,798.5789",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,200.0000",10번_OD경로요금,8.4700,"50,400.0000","87,410.4000","917,608.9789",0.8498,0.1502,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",C_숨은효율,88.5753,360.0000,32.0000,328.0000
25497,11650510_11650520,1165051000,서초1동,1165052000,서초2동,False,7.5000,0.7200,"1,200.0000","79,669.6200",0.0845,0.0678,0.2892,0.0000,5.2000,0.6933,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,고주거비·저통근부담,저부담·지역집중,"779,798.5789",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,200.0000",10번_OD경로요금,5.2500,"50,400.0000","54,180.0000","884,378.5789",0.8817,0.1183,공통시간가치,"10,320.0000",<NA>,1.0000,"643,601.9179","1,069,819.4675","797,133.3682",C_숨은효율,90.1349,329.0000,19.0000,310.0000
27030,11680650_11680640,1168065000,역삼2동,1168064000,역삼1동,False,14.4000,2.1200,"1,500.0000","256,469.6900",0.1789,0.1435,0.1435,0.0000,7.9000,0.5486,API_선택경로요금,TMAP_최종경로,강남구,역삼2동,동남권,6.0000,고주거비·직주근접형,고주거비·저통근부담,저부담·지역집중,"833,857.7797",21.6043,"63,999.6300",0.7294,0.6869,0.0453,22.3300,"1,523.8007","1,500.0000",10번_OD경로요금,10.0800,"63,000.0000","104,025.6000","1,000,883.3797",0.8331,0.1669,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",C_숨은효율,81.2058,384.0000,91.0000,293.0000
25495,11650510_11650530,1165051000,서초1동,1165053000,서초3동,False,10.6000,1.6300,"1,500.0000","136,679.5000",0.1449,0.1164,0.1164,0.0000,6.2000,0.5849,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,고주거비·저통근부담,저부담·지역집중,"779,798.5789",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,500.0000",10번_OD경로요금,7.4200,"63,000.0000","76,574.4000","919,372.9789",0.8482,0.1518,공통시간가치,"10,320.0000",<NA>,1.0000,"639,956.3195","1,050,574.5745","722,860.5682",C_숨은효율,78.6254,358.0000,69.0000,289.0000
27127,11680656_11680640,1168065600,도곡2동,1168064000,역삼1동,False,19.0000,2.7300,"1,500.0000","70,267.7100",0.0782,0.0625,0.1288,0.0000,13.2000,0.6947,API_선택경로요금,TMAP_최종경로,강남구,도곡2동,동남권,2.0000,근접통근 균형형,고주거비·저통근부담,고부담·지역집중,"761,761.8313",28.2494,"64,787.7928",0.6532,0.7624,0.0215,4.8600,"1,542.5665","1,500.0000",10번_OD경로요금,13.3000,"63,000.0000","137,256.0000","962,017.8313",0.7918,0.2082,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",C_숨은효율,84.4865,351.0000,63.0000,288.0000
26281,11650621_11650530,1165062100,방배4동,1165053000,서초3동,False,13.3000,3.3600,"1,500.0000","56,612.4600",0.0687,0.0551,0.1283,1.0000,3.2000,0.2406,API_선택경로요금,TMAP_최종경로,서초구,방배4동,동남권,2.0000,근접통근 균형형,고주거비·저통근부담,저부담·지역집중,"790,735.2330",23.5710,"66,375.4560",0.6585,0.7613,0.0207,15.0500,"1,580.3680","1,500.0000",10번_OD경로요금,9.3100,"63,000.0000","96,079.2000","949,814.4330",0.8325,0.1675,공통시간가치,"10,320.0000",<NA>,1.0000,"639,956.3195","1,050,574.5745","722,860.5682",C_숨은효율,76.1055,368.0000,86.0000,282.0000
27077,11680655_11680640,1168065500,도곡1동,1168064000,역삼1동,False,16.3000,2.1800,"1,500.0000","101,935.7800",0.1150,0.0921,0.0921,0.0000,12.1000,0.7423,API_선택경로요금,TMAP_최종경로,강남구,도곡1동,동남권,2.0000,근접통근 균형형,고주거비·저통근부담,저부담·지역집중,"705,738.5399",25.8483,"64,768.3880",0.7101,0.7186,0.0295,15.7300,"1,542.1045","1,500.0000",10번_OD경로요금,11.4100,"63,000.0000","117,751.2000","886,489.7399",0.7961,0.2039,공통시간가치,"10,320.0000",<NA>,1.0000,"640,717.0543","1,088,169.2296","812,775.0857",C_숨은효율,91.6847,293.0000,13.0000,280.0000
408,11110600_11110615

In [11]:
# =========================================================
# 11. 분석 2 — 주거비 순위 vs 총부담 순위 상관
# =========================================================

rank_corr_rows = []

for work_code, g in efficiency.groupby("근무동코드"):
    rho, p = safe_spearman(
        g["주거비순위"],
        g["총부담순위"],
    )

    rank_corr_rows.append({
        "근무동코드": work_code,
        "후보수": len(g),
        "주거비_총부담_순위Spearman": rho,
        "p_value": p,
        "평균절대순위변화": (
            g["순위변화"].abs().mean()
        ),
        "월세착시비율": (
            g["비용효율유형"] == "B_월세착시"
        ).mean(),
        "숨은효율비율": (
            g["비용효율유형"] == "C_숨은효율"
        ).mean(),
    })

rank_correlation = pd.DataFrame(rank_corr_rows)

display(
    rank_correlation.sort_values(
        "주거비_총부담_순위Spearman"
    ).head(30)
)

,근무동코드,후보수,주거비_총부담_순위Spearman,p_value,평균절대순위변화,월세착시비율,숨은효율비율
130,1129078000,16,0.2647,0.3218,4.5000,0.1875,0.1875
203,1141068500,21,0.3234,0.1527,5.4286,0.2381,0.2381
180,1138053000,16,0.3265,0.2172,4.3750,0.2500,0.2500
61,1120067000,95,0.3381,0.0008,25.2632,0.2105,0.2105
52,1120056000,110,0.3447,0.0002,29.6182,0.2000,0.2000
86,1123060000,60,0.3465,0.0067,15.3667,0.2000,0.2000
118,1129061000,51,0.3521,0.0113,13.4902,0.1961,0.1961
181,1138055100,15,0.3857,0.1556,3.4667,0.2000,0.2000
71,1121578000,45,0.3858,0.0089,11.4222,0.2222,0.2222
400,1171064200,305,0.3921,0.0000,77.2984,0.1902,0.1902


### 분석 2 소득 시나리오

문서의 대표 시나리오를 그대로 비교한다.

- 저소득 청년: 220만 원 / 시간가치 0.3
- 중간소득 청년: 300만 원 / 시간가치 0.5
- 고소득 청년: 400만 원 / 시간가치 0.7

같은 통근시간이라도 소득과 시간가치계수가 높을수록 시간비용이 커진다.

In [12]:
# =========================================================
# 12. 분석 2 — 소득 시나리오별 비용 효율 순위
# =========================================================

SCENARIOS = {
    "저소득_청년": {
        "income": 2_200_000,
        "factor": 0.3,
    },
    "중간소득_청년": {
        "income": 3_000_000,
        "factor": 0.5,
    },
    "고소득_청년": {
        "income": 4_000_000,
        "factor": 0.7,
    },
}

scenario_frames = []

for scenario_name, param in SCENARIOS.items():
    temp = add_burden_metrics(
        analysis,
        monthly_income=param["income"],
        time_value_factor=param["factor"],
    )

    temp = temp[
        temp["편도통근시간_분"] <= MAX_ONEWAY_MINUTES
    ].dropna(
        subset=[
            "표면주거비_원",
            "편도교통비_원",
            "총주거통근부담_원",
        ]
    ).copy()

    temp["시나리오"] = scenario_name
    temp["총부담순위"] = rank_within_group(
        temp,
        "근무동코드",
        "총주거통근부담_원",
        ascending=True,
    )

    scenario_frames.append(temp)

scenario_results = pd.concat(
    scenario_frames,
    ignore_index=True,
)

scenario_top10 = (
    scenario_results[
        scenario_results["총부담순위"] <= TOP_K
    ]
    .sort_values(
        ["시나리오", "근무동코드", "총부담순위"]
    )
)

display(scenario_top10.head(50))


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,4사분면_주거통근,4사분면_부담구조,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,월소득_원,시간가치계수,시나리오,총부담순위
69865,11290580_11110515,1129058000,돈암1동,1111051500,청운효자동,False,29.2000,7.6500,"1,500.0000","1,930.3000",0.0032,0.0026,0.7961,1.0000,9.9000,0.3390,API_선택경로요금,TMAP_최종경로,성북구,돈암제1동,동북권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"432,212.6437",28.8100,"64,575.2066",0.3755,0.8208,0.0139,7.4800,"1,537.5049","1,500.0000",10번_OD경로요금,20.4400,"63,000.0000","273,837.3206","769,049.9643",0.5620,0.4380,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,1.0000
62229,11110680_11110515,1111068000,창신2동,1111051500,청운효자동,False,25.8000,4.5100,"1,500.0000","1,880.7900",0.0065,0.0052,0.7128,1.0000,12.2000,0.4729,API_선택경로요금,TMAP_최종경로,종로구,창신제2동,도심권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·광역분산,"484,720.4861",23.5905,"62,792.2956",0.5142,0.7555,0.0283,10.0000,"1,495.0547","1,500.0000",10번_OD경로요금,18.0600,"63,000.0000","241,952.1531","789,672.6392",0.6138,0.3862,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,2.0000
61775,11110570_11110515,1111057000,무악동,1111051500,청운효자동,False,19.9000,4.2300,"1,500.0000","2,763.9800",0.0155,0.0124,0.5488,1.0000,7.5000,0.3769,API_선택경로요금,TMAP_최종경로,종로구,무악동,도심권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·지역집중,"554,325.3968",22.9649,"64,543.3468",0.6613,0.7437,0.0302,3.3400,"1,536.7464","1,500.0000",10번_OD경로요금,13.9300,"63,000.0000","186,622.0096","803,947.4064",0.6895,0.3105,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,3.0000
61615,11110550_11110515,1111055000,부암동,1111051500,청운효자동,False,11.6000,1.5300,"1,500.0000","17,230.1900",0.0475,0.0381,0.3202,0.0000,8.6000,0.7414,API_선택경로요금,TMAP_최종경로,종로구,부암동,도심권,5.0000,저주거비·지역연계형,고주거비·고통근부담,고부담·지역집중,"645,093.6971",32.5494,"64,874.5332",0.6474,0.7651,0.0245,13.6100,"1,544.6317","1,500.0000",10번_OD경로요금,8.1200,"63,000.0000","108,784.6890","816,878.3861",0.7897,0.2103,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,4.0000
77140,11410520_11110515,1141052000,천연동,1111051500,청운효자동,False,16.9000,2.5000,"1,500.0000","4,624.5300",0.0069,0.0055,0.7226,1.0000,12.3000,0.7278,API_선택경로요금,TMAP_최종경로,서대문구,천연동,서북권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·지역집중,"624,330.9859",24.4663,"62,932.1287",0.6707,0.7501,0.0257,10.4900,"1,498.3840","1,500.0000",10번_OD경로요금,11.8300,"63,000.0000","158,488.0383","845,819.0242",0.7381,0.2619,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,5.0000
62075,11110650_11110515,1111065000,혜화동,1111051500,청운효자동,False,17.2000,4.5800,"1,500.0000","5,814.6200",0.0095,0.0076,0.6105,0.0000,4.7000,0.2733,API_선택경로요금,TMAP_최종경로,종로구,혜화동,도심권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"628,890.2539",24.8816,"62,405.9072",0.3471,0.7332,0.0364,41.3200,"1,485.8549","1,500.0000",10번_OD경로요금,12.0400,"63,000.0000","161,301.4354","853,191.6893",0.7371,0.2629,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,6.0000
70170,11290620_11110515,1129062000,정릉1동,1111051500,청운효자동,False,26.8000,7.1800,"1,500.0000","3,498.8600",0.0062,0.0050,0.6447,0.0000,7.3000,0.2724,API_선택경로요금,TMAP_최종경로,성북구,정릉제1동,동북권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"540,365.6126",30.4062,"64,436.3928",0.3847,0.8238,0.0137,10.8300,"1,534.1998","1,500.0000",10번_OD경로요금,18.7600,"63,000.0000","251,330.1435","854,695.7562",0.6322,0.3678,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,7.0000
77446,11410620_11110515,1141062000,홍제1동,1111051500,청운효자동,False,17.4000,4.8500,"1,550.0000","8,769.8300",0.0079,0.0063,0.6106,1.0000,8.3000,0.4770,API_선택경로요금,TMAP_최종경로,서대문구,홍제제1동,서북권,5.0000,저주거비·지역연계형,고주거비·고통근부담,고부담·지역집중,"657,729.7762",26.8024,"69,097.3323",0.6364,0.7847,0.0181,12.2200,"1,645.1746","1,550.0000",10번_OD경로요금,12.1800,"65,100.0000","163,177.0335","886,006.8097",0.7424,0.2576,월소득환산,"13,397.1292",4000000,0.7000,고소득_청년,8.0000
69677,11290555_11110515,1129055500,삼선동,1111051500,청운효자동,False,17.8000,5.8500,"1,500.0000","3,755.3400",0.0040,0.0032,0.7657,1.0000,1.9000,0.1067,API_선택

# 분석 3. 근무지에 따라 최적 주거지가 어떻게 달라지는가

실행 결과에서 `common["근무동명"]`에는 서울 행정동명이 정상적으로 존재했다.
기존 코드가 잘못된 `근무동명_경로` 컬럼을 조회해 업무지구가 전부 빈 값으로 나온 것이 원인이었다.

이번 버전은 실제 `근무동명`을 사용하고,
`종로1.2.3.4가동`처럼 실제 데이터 표기까지 맞춘다.

비교 조건:
- 주거비 80만 원 이하
- 편도 통근시간 60분 이하

강남처럼 여러 업무동을 묶을 때는 **출근 이동량으로 통근시간·교통비·총부담을 가중평균**한다.

단, 현재 경로 데이터는 거주동별 출근량 누적 80%의 주요 목적지만 조회한 자료이므로,
업무지구별 분석은 서울 427개 완전조합이 아니라 **확보된 주요 통근 OD 범위**에서 해석한다.


In [13]:
# =========================================================
# 13. 분석 3 — 대표 업무지구 정의
# =========================================================

# 실행 결과에서 common["근무동명"]에 실제 행정동명이 정상 존재함을 확인
available_work_names = set(
    common["근무동명"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

# 실제 데이터 표기 기준
BUSINESS_DISTRICTS = {
    "강남": [
        "역삼1동",
        "역삼2동",
        "삼성1동",
        "삼성2동",
        "서초2동",
    ],
    "여의도": [
        "여의동",
    ],
    "광화문·종로": [
        "종로1.2.3.4가동",
        "사직동",
    ],
    "구로·가산": [
        "가산동",
        "구로3동",
    ],
    "상암": [
        "상암동",
    ],
    "잠실": [
        "잠실본동",
        "잠실6동",
        "잠실7동",
    ],
}

district_available = {
    district: [
        name for name in names
        if name in available_work_names
    ]
    for district, names in BUSINESS_DISTRICTS.items()
}

print("[실제 데이터에서 사용 가능한 업무동]")
for district, matched in district_available.items():
    print(
        f"{district:10s} -> "
        f"{len(matched)}개: {matched}"
    )

unmatched_districts = [
    d for d, names in district_available.items()
    if len(names) == 0
]

if unmatched_districts:
    print(
        "\n주의: 실제 근무동이 하나도 매칭되지 않은 업무지구:",
        unmatched_districts,
    )


[실제 데이터에서 사용 가능한 업무동]
강남         -> 5개: ['역삼1동', '역삼2동', '삼성1동', '삼성2동', '서초2동']
여의도        -> 1개: ['여의동']
광화문·종로     -> 2개: ['종로1.2.3.4가동', '사직동']
구로·가산      -> 2개: ['가산동', '구로3동']
상암         -> 1개: ['상암동']
잠실         -> 3개: ['잠실본동', '잠실6동', '잠실7동']


In [14]:
# =========================================================
# 14. 분석 3 — 업무지구별 거주 후보 집계 / Top 10
# =========================================================

def weighted_mean(group, value_col, weight_col="출근_이동량"):
    values = pd.to_numeric(
        group[value_col],
        errors="coerce",
    )

    if (
        weight_col in group.columns
        and group[weight_col].notna().any()
    ):
        weights = pd.to_numeric(
            group[weight_col],
            errors="coerce",
        ).fillna(0)

        valid = values.notna() & (weights > 0)

        if valid.any() and weights[valid].sum() > 0:
            return np.average(
                values[valid],
                weights=weights[valid],
            )

    return values.mean()


district_frames = []
diagnostic_rows = []

for district, work_names in district_available.items():

    if not work_names:
        diagnostic_rows.append({
            "업무지구": district,
            "매칭업무동수": 0,
            "원본OD수": 0,
            "필터후OD수": 0,
            "최종거주후보수": 0,
        })
        continue

    sub = common[
        common["근무동명"].isin(work_names)
    ].copy()

    original_n = len(sub)

    # PDF의 대표 조건
    filtered = sub[
        (sub["편도통근시간_분"] <= SERVICE_MAX_COMMUTE_MINUTES)
        & (sub["표면주거비_원"] <= 800_000)
    ].copy()

    filtered_n = len(filtered)

    if filtered_n == 0:
        diagnostic_rows.append({
            "업무지구": district,
            "매칭업무동수": len(work_names),
            "원본OD수": original_n,
            "필터후OD수": 0,
            "최종거주후보수": 0,
        })
        continue

    rows = []

    for home_code, g in filtered.groupby("거주동코드"):

        first = g.iloc[0]

        row = {
            "거주동코드": home_code,
            "업무지구": district,
            "표면주거비_원": first["표면주거비_원"],
            "편도통근시간_분": weighted_mean(
                g,
                "편도통근시간_분",
            ),
            "월교통비_원": weighted_mean(
                g,
                "월교통비_원",
            ),
            "월통근시간비용_원": weighted_mean(
                g,
                "월통근시간비용_원",
            ),
            "총주거통근부담_원": weighted_mean(
                g,
                "총주거통근부담_원",
            ),
            "업무동포착수": g["근무동코드"].nunique(),
        }

        for col in [
            "거주동명",
            "행정동명",
            "시군구명",
            "권역",
            "행정동_유형",
            "4사분면_주거통근",
            "4사분면_부담구조",
            "청년1인세대_비율",
        ]:
            if col in g.columns:
                row[col] = first[col]

        rows.append(row)

    agg = pd.DataFrame(rows)

    if len(agg):
        agg["추천순위"] = (
            agg["총주거통근부담_원"]
            .rank(
                method="min",
                ascending=True,
            )
        )

        district_frames.append(agg)

    diagnostic_rows.append({
        "업무지구": district,
        "매칭업무동수": len(work_names),
        "원본OD수": original_n,
        "필터후OD수": filtered_n,
        "최종거주후보수": len(agg),
    })


district_diagnostics = pd.DataFrame(
    diagnostic_rows
)

print("[업무지구별 후보 생성 현황]")
display(district_diagnostics)

business_district_results = (
    pd.concat(
        district_frames,
        ignore_index=True,
    )
    if district_frames
    else pd.DataFrame()
)

if len(business_district_results):
    business_top10 = (
        business_district_results[
            business_district_results["추천순위"] <= TOP_K
        ]
        .sort_values(
            ["업무지구", "추천순위"]
        )
        .reset_index(drop=True)
    )

    print("\n[업무지구별 Top 10]")
    display(business_top10)
else:
    business_top10 = pd.DataFrame()
    print("업무지구 분석 결과가 없습니다.")

print(
    "\n주의: 현재 경로 데이터는 거주동별 출근량 누적 80%의 "
    "주요 목적지 OD를 조회한 자료이므로, 업무지구별 후보는 "
    "서울 427개 동의 완전조합이 아니라 '확보된 주요 통근 OD' 범위입니다."
)


[업무지구별 후보 생성 현황]


,업무지구,매칭업무동수,원본OD수,필터후OD수,최종거주후보수
0,강남,5,1967,1400,349
1,여의도,1,425,286,286
2,광화문·종로,2,834,709,371
3,구로·가산,2,791,429,242
4,상암,1,328,214,214
5,잠실,3,441,264,198



[업무지구별 Top 10]


,거주동코드,업무지구,표면주거비_원,편도통근시간_분,월교통비_원,월통근시간비용_원,총주거통근부담_원,업무동포착수,거주동명,행정동명,시군구명,권역,행정동_유형,4사분면_주거통근,4사분면_부담구조,청년1인세대_비율,추천순위
0,1162054500,강남,"502,714.2857",35.2286,"64,369.2442","254,491.0818","821,574.6117",5,청림동,청림동,관악구,서남권,저주거비·지역연계형,저주거비·저통근부담,저부담·광역분산,10.0400,1.0000
1,1126054000,강남,"537,217.3660",32.9414,"67,679.3032","237,968.5332","842,865.2024",5,면목4동,면목제4동,중랑구,동북권,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,11.8900,2.0000
2,1126057500,강남,"517,404.3792",36.4143,"67,340.4045","263,056.7241","847,801.5077",5,면목3.8동,면목제3.8동,중랑구,동북권,광역분산 통근형,저주거비·고통근부담,저부담·광역분산,15.5800,3.0000
3,1162074500,강남,"472,036.5682",43.7884,"68,792.9722","316,327.6722","857,157.2126",5,삼성동,삼성동,관악구,서남권,저주거비·지역연계형,저주거비·저통근부담,저부담·광역분산,7.4600,4.0000
4,1168066000,강남,"640,717.0543",21.8186,"58,880.5887","157,617.6650","857,215.3079",5,개포1동,개포1동,강남구,동남권,저주거비·지역연계형,저주거비·저통근부담,저부담·지역집중,2.6000,5.0000
5,1171051000,강남,"478,765.4867",43.6433,"65,380.6428","315,279.5218","859,425.6514",5,풍납1동,풍납1동,송파구,동남권,저주거비·지역연계형,저주거비·저통근부담,저부담·지역집중,9.4100,6.0000
6,1159067000,강남,"516,698.3806",37.9984,"70,401.3966","274,500.7295","861,600.5066",5,신대방1동,신대방제1동,동작구,서남권,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,17.9400,7.0000
7,1159051000,강남,"522,067.1488",34.7533,"94,893.6533","251,057.6054","868,018.4074",5,노량진1동,노량진제1동,동작구,서남권,근접통근 균형형,저주거비·저통근부담,저부담·지역집중,33.6400,8.0000
8,1120058000,강남,"562,384.6154",34.5202,"63,229.8815","249,374.1612","874,988.6581",5,응봉동,응봉동,성동구,동북권,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,7.2600,9.0000
9,1120072000,강남,"582,179.4977",31.8755,"63,314.1741","230,268.3885","875,762.0603",5,송정동,송정동,성동구,동북권,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,22.4800,10.0000



주의: 현재 경로 데이터는 거주동별 출근량 누적 80%의 주요 목적지 OD를 조회한 자료이므로, 업무지구별 후보는 서울 427개 동의 완전조합이 아니라 '확보된 주요 통근 OD' 범위입니다.


In [15]:
# =========================================================
# 15. 분석 3 — Top10 중복률 / 추천 1순위 비교
# =========================================================

OVERLAP_COLUMNS = [
    "업무지구1",
    "업무지구2",
    "업무지구1_Top후보수",
    "업무지구2_Top후보수",
    "공통동수",
    "Top10_중복률",
    "공통거주동코드",
]

overlap_rows = []

if len(business_top10):

    districts = sorted(
        business_top10["업무지구"]
        .dropna()
        .unique()
    )

    print("비교 가능한 업무지구:", districts)

    for i in range(len(districts)):
        for j in range(i + 1, len(districts)):

            d1 = districts[i]
            d2 = districts[j]

            top1 = (
                business_top10[
                    business_top10["업무지구"] == d1
                ]
                .sort_values("추천순위")
                ["거주동코드"]
                .head(TOP_K)
                .tolist()
            )

            top2 = (
                business_top10[
                    business_top10["업무지구"] == d2
                ]
                .sort_values("추천순위")
                ["거주동코드"]
                .head(TOP_K)
                .tolist()
            )

            common_codes = set(top1) & set(top2)

            denominator = min(
                TOP_K,
                len(top1),
                len(top2),
            )

            overlap_rows.append({
                "업무지구1": d1,
                "업무지구2": d2,
                "업무지구1_Top후보수": len(top1),
                "업무지구2_Top후보수": len(top2),
                "공통동수": len(common_codes),
                "Top10_중복률": (
                    len(common_codes) / denominator
                    if denominator > 0
                    else np.nan
                ),
                "공통거주동코드": " | ".join(
                    sorted(common_codes)
                ),
            })


district_top10_overlap = pd.DataFrame(
    overlap_rows,
    columns=OVERLAP_COLUMNS,
)

if len(district_top10_overlap):
    display(
        district_top10_overlap.sort_values(
            "Top10_중복률"
        )
    )
else:
    print(
        "업무지구 간 Top10 중복률을 계산할 수 없습니다."
    )


# 업무지구별 추천 1순위
if len(business_top10):
    district_top1 = (
        business_top10
        .sort_values(["업무지구", "추천순위"])
        .groupby("업무지구")
        .head(1)
        [
            [
                "업무지구",
                "거주동코드",
                "거주동명",
                "행정동_유형",
                "추천순위",
                "총주거통근부담_원",
                "편도통근시간_분",
                "표면주거비_원",
            ]
        ]
        .reset_index(drop=True)
    )

    print("\n[업무지구별 추천 1순위]")
    display(district_top1)
else:
    district_top1 = pd.DataFrame()


비교 가능한 업무지구: ['강남', '광화문·종로', '구로·가산', '상암', '여의도', '잠실']


,업무지구1,업무지구2,업무지구1_Top후보수,업무지구2_Top후보수,공통동수,Top10_중복률,공통거주동코드
0,강남,광화문·종로,10,10,0,0.0000,
2,강남,상암,10,10,0,0.0000,
6,광화문·종로,상암,10,10,0,0.0000,
11,구로·가산,잠실,10,10,0,0.0000,
13,상암,잠실,10,10,0,0.0000,
14,여의도,잠실,10,10,0,0.0000,
1,강남,구로·가산,10,10,1,0.1000,1159067000
4,강남,잠실,10,10,1,0.1000,1171051000
5,광화문·종로,구로·가산,10,10,1,0.1000,1156065000
9,구로·가산,상암,10,10,1,0.1000,1153073000



[업무지구별 추천 1순위]


,업무지구,거주동코드,거주동명,행정동_유형,추천순위,총주거통근부담_원,편도통근시간_분,표면주거비_원
0,강남,1162054500,청림동,저주거비·지역연계형,1.0000,"821,574.6117",35.2286,"502,714.2857"
1,광화문·종로,1111068000,창신2동,근접통근 균형형,1.0000,"692,474.8163",20.0380,"484,720.4861"
2,구로·가산,1156067000,신길5동,저주거비·지역연계형,1.0000,"663,450.9950",18.4514,"465,914.0893"
3,상암,1138064000,증산동,저주거비·지역연계형,1.0000,"702,906.9377",16.3000,"522,155.7377"
4,여의도,1156065000,신길3동,저주거비·지역연계형,1.0000,"647,862.4641",14.9000,"477,224.8641"
5,잠실,1171051000,풍납1동,저주거비·지역연계형,1.0000,"699,537.9417",21.0811,"478,765.4867"


In [16]:
# =========================================================
# 15-1. 업무지구 분석 데이터 품질 확인
# =========================================================

if len(district_diagnostics):
    valid_districts = district_diagnostics[
        district_diagnostics["최종거주후보수"] > 0
    ]

    print(
        "후보가 생성된 업무지구:",
        valid_districts["업무지구"].tolist(),
    )

    print(
        "Top10 비교가 가능한 업무지구 수:",
        int(
            (
                valid_districts["최종거주후보수"]
                >= TOP_K
            ).sum()
        ),
    )

    display(
        valid_districts.sort_values(
            "최종거주후보수",
            ascending=False,
        )
    )


후보가 생성된 업무지구: ['강남', '여의도', '광화문·종로', '구로·가산', '상암', '잠실']
Top10 비교가 가능한 업무지구 수: 6


,업무지구,매칭업무동수,원본OD수,필터후OD수,최종거주후보수
2,광화문·종로,2,834,709,371
0,강남,5,1967,1400,349
1,여의도,1,425,286,286
3,구로·가산,2,791,429,242
4,상암,1,328,214,214
5,잠실,3,441,264,198


In [17]:
# =========================================================
# 16. 분석 3 — 순위 변화 / 업무지구 특화도
# =========================================================

if len(business_district_results):

    rank_pivot = (
        business_district_results
        .pivot_table(
            index="거주동코드",
            columns="업무지구",
            values="추천순위",
            aggfunc="first",
        )
    )

    burden_pivot = (
        business_district_results
        .pivot_table(
            index="거주동코드",
            columns="업무지구",
            values="총주거통근부담_원",
            aggfunc="first",
        )
    )

    specialization_rows = []

    for district in burden_pivot.columns:

        others = [
            c for c in burden_pivot.columns
            if c != district
        ]

        if not others:
            continue

        other_mean = burden_pivot[others].mean(
            axis=1,
            skipna=True,
        )

        score = (
            other_mean
            - burden_pivot[district]
        )

        valid = (
            burden_pivot[district].notna()
            & other_mean.notna()
        )

        temp = pd.DataFrame({
            "거주동코드": burden_pivot.index[valid],
            "업무지구": district,
            "업무지구특화도_원": score[valid].values,
        })

        specialization_rows.append(temp)

    district_specialization = (
        pd.concat(
            specialization_rows,
            ignore_index=True,
        )
        if specialization_rows
        else pd.DataFrame(
            columns=[
                "거주동코드",
                "업무지구",
                "업무지구특화도_원",
            ]
        )
    )

    if len(district_specialization):
        display(
            district_specialization.sort_values(
                "업무지구특화도_원",
                ascending=False,
            ).head(50)
        )

    print("[업무지구별 순위 피벗]")
    display(rank_pivot.head(30))

else:
    rank_pivot = pd.DataFrame()
    district_specialization = pd.DataFrame(
        columns=[
            "거주동코드",
            "업무지구",
            "업무지구특화도_원",
        ]
    )


,거주동코드,업무지구,업무지구특화도_원
1625,1171069000,잠실,"311,797.5724"
848,1153059500,구로·가산,"289,154.7846"
355,1111060000,광화문·종로,"273,535.3248"
358,1111064000,광화문·종로,"273,131.8134"
288,1165051000,강남,"270,117.4050"
1618,1171061000,잠실,"262,096.5068"
359,1111065000,광화문·종로,"258,572.0615"
1624,1171067000,잠실,"257,311.4179"
846,1153055000,구로·가산,"256,711.4615"
357,1111063000,광화문·종로,"256,570.7914"


[업무지구별 순위 피벗]


업무지구,강남,광화문·종로,구로·가산,상암,여의도,잠실
거주동코드,,,,,,
1111051500,320.0000,107.0000,177.0000,170.0000,202.0000,NaN
1111054000,326.0000,110.0000,211.0000,NaN,196.0000,NaN
1111055000,306.0000,85.0000,163.0000,179.0000,190.0000,NaN
1111056000,334.0000,190.0000,214.0000,146.0000,237.0000,NaN
1111057000,79.0000,3.0000,87.0000,NaN,68.0000,NaN
1111058000,298.0000,69.0000,143.0000,188.0000,182.0000,NaN
1111060000,304.0000,77.0000,229.0000,192.0000,264.0000,NaN
1111061500,309.0000,183.0000,232.0000,NaN,193.0000,NaN
1111063000,295.0000,81.0000,176.0000,NaN,270.0000,NaN


### 분석 3 권역 단위 요약

동 단위 추천만 보여주면 인접 동이 연속으로 나올 수 있으므로,
14_2 결과에 `권역`이 있다면 **권역 → 세부 추천동**의 2단계로 요약한다.

권역이 없다면 이 셀은 자동으로 건너뛴다.

In [18]:
# =========================================================
# 17. 분석 3 — 권역 단위 비용 효율 요약
# =========================================================

if (
    len(business_district_results)
    and "권역" in business_district_results.columns
):
    district_region_summary = (
        business_district_results
        .dropna(subset=["권역"])
        .groupby(["업무지구", "권역"])
        .agg(
            후보동수=("거주동코드", "count"),
            평균총부담_원=("총주거통근부담_원", "mean"),
            평균편도통근시간_분=("편도통근시간_분", "mean"),
            평균표면주거비_원=("표면주거비_원", "mean"),
        )
        .reset_index()
    )

    district_region_summary["권역순위"] = (
        district_region_summary
        .groupby("업무지구")["평균총부담_원"]
        .rank(method="min")
    )

    display(
        district_region_summary.sort_values(
            ["업무지구", "권역순위"]
        )
    )
else:
    district_region_summary = pd.DataFrame()
    print("권역 컬럼이 없어 권역 분석은 건너뜁니다.")

,업무지구,권역,후보동수,평균총부담_원,평균편도통근시간_분,평균표면주거비_원,권역순위
1,강남,동남권,61,"1,001,269.2396",36.0102,"673,048.2888",1.0000
3,강남,서남권,106,"1,018,140.9619",46.6424,"600,187.1121",2.0000
2,강남,동북권,107,"1,018,298.7062",46.4869,"609,973.2757",3.0000
0,강남,도심권,35,"1,052,704.8824",42.0394,"682,827.6290",4.0000
4,강남,서북권,40,"1,105,993.2844",53.0271,"641,208.5391",5.0000
5,광화문·종로,도심권,35,"899,929.1985",21.3134,"682,827.6290",1.0000
9,광화문·종로,서북권,43,"931,434.8912",31.5970,"638,687.3848",2.0000
7,광화문·종로,동북권,123,"943,547.5228",38.4710,"599,370.9724",3.0000
8,광화문·종로,서남권,109,"995,424.6634",45.1769,"598,147.3583",4.0000
6,광화문·종로,동남권,61,"1,079,864.0168",46.2916,"673,048.2888",5.0000


# 분석 4. 추천 결과를 결정하는 핵심 조건은 무엇인가

추천점수는 비용형 변수를 모두 **낮을수록 좋은 0~100점**으로 변환한 뒤 가중합한다.

기본 가중치:
- 주거비 40%
- 통근시간 30%
- 교통비 10%

현재 프로젝트 데이터에 생활비/주거환경 점수가 있으면 확장할 수 있지만,
없다면 임의로 만들지 않고 이 세 변수만으로 분석한다.

이 분석의 핵심은 회귀보다:
- 기여도
- 가중치 변화
- 소득 변화
- 예산 변화
- 추천 순위 안정성
- 추천 1순위 전환점

이다.

In [19]:
# =========================================================
# 18. 분석 4 — 추천점수 함수
# =========================================================

BASE_WEIGHTS = {
    "housing": 0.40,
    "commute_time": 0.30,
    "transport": 0.10,
}

# 현재 사용 가능한 변수만 정규화해서 가중치 합을 1로 조정
weight_sum = sum(BASE_WEIGHTS.values())
BASE_WEIGHTS = {
    k: v / weight_sum
    for k, v in BASE_WEIGHTS.items()
}

print("실사용 기본 가중치:", BASE_WEIGHTS)


def add_recommendation_score(
    df,
    housing_weight=BASE_WEIGHTS["housing"],
    commute_weight=BASE_WEIGHTS["commute_time"],
    transport_weight=BASE_WEIGHTS["transport"],
):
    out = df.copy()

    out["주거비점수"] = out.groupby(
        "근무동코드",
        group_keys=False,
    )["표면주거비_원"].transform(
        minmax_good_low
    )

    out["통근시간점수"] = out.groupby(
        "근무동코드",
        group_keys=False,
    )["편도통근시간_분"].transform(
        minmax_good_low
    )

    out["교통비점수"] = out.groupby(
        "근무동코드",
        group_keys=False,
    )["월교통비_원"].transform(
        minmax_good_low
    )

    total_w = (
        housing_weight
        + commute_weight
        + transport_weight
    )

    housing_weight /= total_w
    commute_weight /= total_w
    transport_weight /= total_w

    out["주거비기여도"] = (
        out["주거비점수"] * housing_weight
    )
    out["통근시간기여도"] = (
        out["통근시간점수"] * commute_weight
    )
    out["교통비기여도"] = (
        out["교통비점수"] * transport_weight
    )

    out["추천점수"] = (
        out["주거비기여도"]
        + out["통근시간기여도"]
        + out["교통비기여도"]
    )

    contribution_sum = (
        out["주거비기여도"]
        + out["통근시간기여도"]
        + out["교통비기여도"]
    ).replace(0, np.nan)

    out["주거비기여비중"] = (
        out["주거비기여도"]
        / contribution_sum
    )
    out["통근시간기여비중"] = (
        out["통근시간기여도"]
        / contribution_sum
    )
    out["교통비기여비중"] = (
        out["교통비기여도"]
        / contribution_sum
    )

    out["추천순위"] = (
        out.groupby("근무동코드")["추천점수"]
        .rank(
            method="min",
            ascending=False,
        )
    )

    return out


recommendation_base = add_recommendation_score(
    common
)

display(
    recommendation_base[
        recommendation_base["추천순위"] <= 10
    ].sort_values(
        ["근무동코드", "추천순위"]
    ).head(50)
)

실사용 기본 가중치: {'housing': 0.5, 'commute_time': 0.37499999999999994, 'transport': 0.125}


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,4사분면_주거통근,4사분면_부담구조,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,월소득_원,시간가치계수,주거비점수,통근시간점수,교통비점수,주거비기여도,통근시간기여도,교통비기여도,추천점수,주거비기여비중,통근시간기여비중,교통비기여비중,추천순위
8475,11290580_11110515,1129058000,돈암1동,1111051500,청운효자동,False,29.2000,7.6500,"1,500.0000","1,930.3000",0.0032,0.0026,0.7961,1.0000,9.9000,0.3390,API_선택경로요금,TMAP_최종경로,성북구,돈암제1동,동북권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"432,212.6437",28.8100,"64,575.2066",0.3755,0.8208,0.0139,7.4800,"1,537.5049","1,500.0000",10번_OD경로요금,20.4400,"63,000.0000","210,940.8000","706,153.4437",0.6121,0.3879,공통시간가치,"10,320.0000",<NA>,1.0000,100.0000,59.5402,71.4286,50.0000,22.3276,8.9286,81.2562,0.6153,0.2748,0.1099,1.0000
761,11110680_11110515,1111068000,창신2동,1111051500,청운효자동,False,25.8000,4.5100,"1,500.0000","1,880.7900",0.0065,0.0052,0.7128,1.0000,12.2000,0.4729,API_선택경로요금,TMAP_최종경로,종로구,창신제2동,도심권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·광역분산,"484,720.4861",23.5905,"62,792.2956",0.5142,0.7555,0.0283,10.0000,"1,495.0547","1,500.0000",10번_OD경로요금,18.0600,"63,000.0000","186,379.2000","734,099.6861",0.6603,0.3397,공통시간가치,"10,320.0000",<NA>,1.0000,89.7293,67.3563,71.4286,44.8647,25.2586,8.9286,79.0519,0.5675,0.3195,0.1129,2.0000
307,11110570_11110515,1111057000,무악동,1111051500,청운효자동,False,19.9000,4.2300,"1,500.0000","2,763.9800",0.0155,0.0124,0.5488,1.0000,7.5000,0.3769,API_선택경로요금,TMAP_최종경로,종로구,무악동,도심권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·지역집중,"554,325.3968",22.9649,"64,543.3468",0.6613,0.7437,0.0302,3.3400,"1,536.7464","1,500.0000",10번_OD경로요금,13.9300,"63,000.0000","143,757.6000","761,082.9968",0.7283,0.2717,공통시간가치,"10,320.0000",<NA>,1.0000,76.1145,80.9195,71.4286,38.0572,30.3448,8.9286,77.3306,0.4921,0.3924,0.1155,3.0000
147,11110550_11110515,1111055000,부암동,1111051500,청운효자동,False,11.6000,1.5300,"1,500.0000","17,230.1900",0.0475,0.0381,0.3202,0.0000,8.6000,0.7414,API_선택경로요금,TMAP_최종경로,종로구,부암동,도심권,5.0000,저주거비·지역연계형,고주거비·고통근부담,고부담·지역집중,"645,093.6971",32.5494,"64,874.5332",0.6474,0.7651,0.0245,13.6100,"1,544.6317","1,500.0000",10번_OD경로요금,8.1200,"63,000.0000","83,798.4000","791,892.0971",0.8146,0.1854,공통시간가치,"10,320.0000",<NA>,1.0000,58.3600,100.0000,71.4286,29.1800,37.5000,8.9286,75.6086,0.3859,0.4960,0.1181,4.0000
15776,11410520_11110515,1141052000,천연동,1111051500,청운효자동,False,16.9000,2.5000,"1,500.0000","4,624.5300",0.0069,0.0055,0.7226,1.0000,12.3000,0.7278,API_선택경로요금,TMAP_최종경로,서대문구,천연동,서북권,2.0000,근접통근 균형형,저주거비·저통근부담,저부담·지역집중,"624,330.9859",24.4663,"62,932.1287",0.6707,0.7501,0.0257,10.4900,"1,498.3840","1,500.0000",10번_OD경로요금,11.8300,"63,000.0000","122,085.6000","809,416.5859",0.7713,0.2287,공통시간가치,"10,320.0000",<NA>,1.0000,62.4212,87.8161,71.4286,31.2106,32.9310,8.9286,73.0702,0.4271,0.4507,0.1222,5.0000
8780,11290620_11110515,1129062000,정릉1동,1111051500,청운효자동,False,26.8000,7.1800,"1,500.0000","3,498.8600",0.0062,0.0050,0.6447,0.0000,7.3000,0.2724,API_선택경로요금,TMAP_최종경로,성북구,정릉제1동,동북권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"540,365.6126",30.4062,"64,436.3928",0.3847,0.8238,0.0137,10.8300,"1,534.1998","1,500.0000",10번_OD경로요금,18.7600,"63,000.0000","193,603.2000","796,968.8126",0.6780,0.3220,공통시간가치,"10,320.0000",<NA>,1.0000,78.8450,65.0575,71.4286,39.4225,24.3966,8.9286,72.7476,0.5419,0.3354,0.1227,6.0000
607,11110650_11110515,1111065000,혜화동,1111051500,청운효자동,False,17.2000,4.5800,"1,500.0000","5,814.6200",0.0095,0.0076,0.6105,0.0000,4.7000,0.2733,API_선택경로요금,TMAP_최종경로,종로구,혜화동,도심권,3.0000,광역분산 통근형,저주거비·저통근부담,저부담·광역분산,"628,890.2539",24.8816,"62,405.9072",0.3471,0.7332,0.0364,41.3200,"1,485.8549","1,500.0000",10번_OD경로요금,12.0400,"63,000.0000","124,252.8000","816,143.0539",0.7706,0.2294,공통시간가치,"10,320.0000",<NA>,1.0000,61.5294,87.1264,71.4286,30.7647,32.6724,8.9286,72.3657,0.4251,0.4515,0.1234,7.0000
8287,11290555_11110515,1129055500,삼선동,1111051500,청운효자동,False,17.

In [20]:
# =========================================================
# 19. 분석 4 — 가중치 민감도 / 추천 전환점
# =========================================================

HOUSING_WEIGHT_GRID = np.arange(
    0.30,
    0.501,
    0.05,
)

weight_sensitivity_rows = []

# 나머지 통근시간/교통비는 기존 비율 3:1 유지
remaining_ratio_time = 0.75
remaining_ratio_transport = 0.25

for housing_w in HOUSING_WEIGHT_GRID:
    remain = 1 - housing_w

    commute_w = remain * remaining_ratio_time
    transport_w = remain * remaining_ratio_transport

    temp = add_recommendation_score(
        common,
        housing_weight=housing_w,
        commute_weight=commute_w,
        transport_weight=transport_w,
    )

    top = (
        temp.sort_values(
            ["근무동코드", "추천순위"]
        )
        .groupby("근무동코드")
        .head(5)
    )

    for _, row in top.iterrows():
        weight_sensitivity_rows.append({
            "주거비가중치": housing_w,
            "통근시간가중치": commute_w,
            "교통비가중치": transport_w,
            "근무동코드": row["근무동코드"],
            "거주동코드": row["거주동코드"],
            "행정동명": row.get("행정동명", pd.NA),
            "추천순위": row["추천순위"],
            "추천점수": row["추천점수"],
        })

weight_sensitivity = pd.DataFrame(
    weight_sensitivity_rows
)

# 근무동별 1순위가 처음 바뀌는 가중치
weight_top1 = (
    weight_sensitivity[
        weight_sensitivity["추천순위"] == 1
    ]
    .sort_values(
        ["근무동코드", "주거비가중치"]
    )
)

weight_switch_rows = []

for work_code, g in weight_top1.groupby("근무동코드"):
    g = g.sort_values("주거비가중치")
    base_home = g.iloc[0]["거주동코드"]

    changed = g[
        g["거주동코드"] != base_home
    ]

    if len(changed):
        first = changed.iloc[0]

        weight_switch_rows.append({
            "근무동코드": work_code,
            "기준1순위": base_home,
            "전환주거비가중치": first["주거비가중치"],
            "전환후1순위": first["거주동코드"],
        })

weight_switchpoints = pd.DataFrame(
    weight_switch_rows
)

display(weight_switchpoints.head(50))

,근무동코드,기준1순위,전환주거비가중치,전환후1순위
0,1111051500,1111055000,0.4500,1129058000
1,1111053000,1111057000,0.5000,1111068000
2,1111054000,1111051500,0.3500,1129058000
3,1111055000,1111051500,0.4500,1129064000
4,1111056000,1111055000,0.4500,1129065000
5,1111058000,1141052000,0.4500,1111057000
6,1111060000,1111054000,0.5000,1111057000
7,1111061500,1111060000,0.4000,1111068000
8,1111065000,1111063000,0.4500,1129058000
9,1111068000,1111067000,0.4500,1111069000


In [21]:
# =========================================================
# 20. 분석 4 — 소득 민감도 / 전환점
# =========================================================

INCOME_GRID = np.arange(
    2_000_000,
    4_000_001,
    100_000,
)

income_rows = []

for income in INCOME_GRID:

    # 소득 민감도에서만 소득 기반 시간가치 사용
    temp = add_burden_metrics(
        analysis,
        monthly_income=income,
        time_value_factor=0.5,
    )

    temp = temp[
        temp["편도통근시간_분"] <= MAX_ONEWAY_MINUTES
    ].dropna(
        subset=[
            "표면주거비_원",
            "편도교통비_원",
            "총주거통근부담_원",
        ]
    ).copy()

    temp["총부담순위"] = rank_within_group(
        temp,
        "근무동코드",
        "총주거통근부담_원",
        ascending=True,
    )

    top1 = (
        temp.sort_values(
            ["근무동코드", "총부담순위"]
        )
        .groupby("근무동코드")
        .head(1)
    )

    for _, row in top1.iterrows():
        income_rows.append({
            "월소득_원": income,
            "근무동코드": row["근무동코드"],
            "근무동명": row.get("근무동명", pd.NA),
            "1순위거주동코드": row["거주동코드"],
            "1순위행정동명": row.get("거주동명", pd.NA),
            "행정동_유형": row.get("행정동_유형", pd.NA),
            "총부담_원": row["총주거통근부담_원"],
            "편도통근시간_분": row["편도통근시간_분"],
            "표면주거비_원": row["표면주거비_원"],
        })

income_sensitivity = pd.DataFrame(income_rows)

income_switch_rows = []

if len(income_sensitivity):

    for work_code, g in income_sensitivity.groupby(
        "근무동코드"
    ):

        g = g.sort_values("월소득_원")
        previous_home = None

        for _, row in g.iterrows():

            current_home = row[
                "1순위거주동코드"
            ]

            if (
                previous_home is not None
                and current_home != previous_home
            ):
                income_switch_rows.append({
                    "근무동코드": work_code,
                    "전환소득_원": row["월소득_원"],
                    "이전1순위": previous_home,
                    "변경1순위": current_home,
                })

            previous_home = current_home

income_switchpoints = pd.DataFrame(
    income_switch_rows,
    columns=[
        "근무동코드",
        "전환소득_원",
        "이전1순위",
        "변경1순위",
    ],
)

print("[소득 변화에 따른 1순위 전환점]")
display(income_switchpoints.head(50))


[소득 변화에 따른 1순위 전환점]


,근무동코드,전환소득_원,이전1순위,변경1순위
0,1111053000,2600000,1129058000,1111068000
1,1111057000,2600000,1141064000,1141052000
2,1111058000,3200000,1129058000,1111057000
3,1111063000,2300000,1129058000,1111068000
4,1111064000,2100000,1129058000,1111068000
5,1111067000,2600000,1129058000,1111068000
6,1114052000,3500000,1129058000,1111068000
7,1114052000,3800000,1111068000,1111057000
8,1114060500,3100000,1129058000,1111068000
9,1114061500,2400000,1129058000,1111068000


In [22]:
# =========================================================
# 21. 분석 4 — 예산 민감도 / 추가 10만원 효과
# =========================================================

BUDGET_GRID = [
    600_000,
    700_000,
    800_000,
    900_000,
    1_000_000,
]

budget_rows = []

for budget in BUDGET_GRID:
    temp = common[
        (common["표면주거비_원"] <= budget)
        & (
            common["편도통근시간_분"]
            <= SERVICE_MAX_COMMUTE_MINUTES
        )
    ].copy()

    if len(temp) == 0:
        continue

    temp["총부담순위"] = rank_within_group(
        temp,
        "근무동코드",
        "총주거통근부담_원",
        ascending=True,
    )

    top1 = temp[
        temp["총부담순위"] == 1
    ]

    for _, row in top1.iterrows():
        budget_rows.append({
            "주거비예산상한_원": budget,
            "근무동코드": row["근무동코드"],
            "1순위거주동코드": row["거주동코드"],
            "1순위행정동명": row.get("행정동명", pd.NA),
            "편도통근시간_분": row["편도통근시간_분"],
            "총부담_원": row["총주거통근부담_원"],
            "표면주거비_원": row["표면주거비_원"],
        })

budget_sensitivity = pd.DataFrame(
    budget_rows
)

budget_effect_rows = []

for work_code, g in budget_sensitivity.groupby("근무동코드"):
    g = g.sort_values("주거비예산상한_원").reset_index(drop=True)

    for i in range(1, len(g)):
        prev = g.iloc[i - 1]
        cur = g.iloc[i]

        budget_effect_rows.append({
            "근무동코드": work_code,
            "이전예산_원": prev["주거비예산상한_원"],
            "변경예산_원": cur["주거비예산상한_원"],
            "이전1순위": prev["1순위거주동코드"],
            "변경1순위": cur["1순위거주동코드"],
            "추가예산_원": (
                cur["주거비예산상한_원"]
                - prev["주거비예산상한_원"]
            ),
            "편도통근시간절감_분": (
                prev["편도통근시간_분"]
                - cur["편도통근시간_분"]
            ),
            "총부담변화_원": (
                cur["총부담_원"]
                - prev["총부담_원"]
            ),
        })

budget_effect = pd.DataFrame(
    budget_effect_rows
)

display(budget_effect.head(50))

,근무동코드,이전예산_원,변경예산_원,이전1순위,변경1순위,추가예산_원,편도통근시간절감_분,총부담변화_원
0,1111051500,600000,700000,1129058000,1129058000,100000,0.0000,0.0000
1,1111051500,700000,800000,1129058000,1129058000,100000,0.0000,0.0000
2,1111051500,800000,900000,1129058000,1129058000,100000,0.0000,0.0000
3,1111051500,900000,1000000,1129058000,1129058000,100000,0.0000,0.0000
4,1111053000,600000,700000,1111068000,1111068000,100000,0.0000,0.0000
5,1111053000,700000,800000,1111068000,1111068000,100000,0.0000,0.0000
6,1111053000,800000,900000,1111068000,1111068000,100000,0.0000,0.0000
7,1111053000,900000,1000000,1111068000,1111068000,100000,0.0000,0.0000
8,1111054000,600000,700000,1129058000,1129058000,100000,0.0000,0.0000
9,1111054000,700000,800000,1129058000,1129058000,100000,0.0000,0.0000


### 추천 안정성

조건 변화 전후에:
- Top 5 중복률
- 전체 순위 Spearman 상관
- 평균 절대 순위변동

을 계산한다.

작은 조건 변화에도 이 값들이 크게 흔들리면 추천이 민감하다는 뜻이다.

In [23]:
# =========================================================
# 22. 분석 4 — 추천 안정성
# =========================================================

baseline_rank = recommendation_base[
    [
        "근무동코드",
        "거주동코드",
        "추천순위",
    ]
].rename(
    columns={"추천순위": "기준순위"}
)

stability_rows = []

for housing_w in HOUSING_WEIGHT_GRID:
    remain = 1 - housing_w

    temp = add_recommendation_score(
        common,
        housing_weight=housing_w,
        commute_weight=remain * remaining_ratio_time,
        transport_weight=remain * remaining_ratio_transport,
    )

    merged = baseline_rank.merge(
        temp[
            [
                "근무동코드",
                "거주동코드",
                "추천순위",
            ]
        ],
        on=["근무동코드", "거주동코드"],
        how="inner",
    )

    for work_code, g in merged.groupby("근무동코드"):
        rho, p = safe_spearman(
            g["기준순위"],
            g["추천순위"],
        )

        base_top5 = (
            g.nsmallest(5, "기준순위")["거주동코드"]
            .tolist()
        )
        new_top5 = (
            g.nsmallest(5, "추천순위")["거주동코드"]
            .tolist()
        )

        stability_rows.append({
            "주거비가중치": housing_w,
            "근무동코드": work_code,
            "Top5_중복률": (
                len(set(base_top5) & set(new_top5)) / 5
            ),
            "Spearman_순위상관": rho,
            "Spearman_p": p,
            "평균절대순위변동": (
                g["추천순위"]
                .sub(g["기준순위"])
                .abs()
                .mean()
            ),
        })

recommendation_stability = pd.DataFrame(
    stability_rows
)

display(
    recommendation_stability.sort_values(
        ["주거비가중치", "평균절대순위변동"],
        ascending=[True, False],
    ).head(50)
)

,주거비가중치,근무동코드,Top5_중복률,Spearman_순위상관,Spearman_p,평균절대순위변동
370,0.3000,1168064000,0.4000,0.8903,0.0000,43.9391
344,0.3000,1165053000,0.4000,0.8901,0.0000,43.0827
343,0.3000,1165052000,0.4000,0.8839,0.0000,42.7500
18,0.3000,1114054000,0.6000,0.8919,0.0000,42.1981
365,0.3000,1168058000,0.4000,0.9011,0.0000,41.6960
196,0.3000,1141056500,0.6000,0.8682,0.0000,40.8285
8,0.3000,1111061500,0.4000,0.9021,0.0000,40.5070
19,0.3000,1114055000,0.6000,0.9041,0.0000,40.2389
17,0.3000,1114052000,0.4000,0.8981,0.0000,40.0751
362,0.3000,1168053100,0.4000,0.9074,0.0000,39.6930


# 최종 산출물

이 노트북에서 반드시 확보하는 핵심 결과표:

1. **주거비–통근 교환관계**
   - 10만 원 절감당 통근시간/교통비/시간비용 증가
   - 실질 순절감액
   - 손익분기 주거비 절감액
2. **실질 비용 효율**
   - 월세 순위 vs 총부담 순위
   - 월세 착시 / 숨은 효율
   - 비용 구성비
3. **근무지 효과**
   - 업무지구별 Top 10
   - Top 10 중복률
   - 순위 변화 / 특화도 / 권역
4. **개인 조건별 전환점**
   - 가중치 전환점
   - 소득 전환점
   - 예산 변화 효과
   - 추천 안정성

14_2의 6개 유형과 4사분면은 각 결과를 설명하는 메타정보로 같이 보존한다.

In [24]:
# =========================================================
# 23. 최종 결과 저장
# =========================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUTS = {
    "common_table":
        PROCESSED_DIR / "insight_common_work_home_table.csv",

    "tradeoff_all_cheaper":
        PROCESSED_DIR / "insight_housing_commute_tradeoff_all_cheaper.csv",

    "tradeoff_100k_detail":
        PROCESSED_DIR / "insight_housing_commute_tradeoff_100k_detail.csv",

    "tradeoff_summary":
        PROCESSED_DIR / "insight_housing_commute_tradeoff_summary.csv",

    "tradeoff_regression":
        PROCESSED_DIR / "insight_housing_commute_tradeoff_regression.csv",

    "housing_band":
        PROCESSED_DIR / "insight_housing_band_commute_summary.csv",

    "efficiency":
        PROCESSED_DIR / "insight_cost_efficiency_by_work_home.csv",

    "rank_correlation":
        PROCESSED_DIR / "insight_housing_vs_total_rank_correlation.csv",

    "scenario_top10":
        PROCESSED_DIR / "insight_income_scenario_top10.csv",

    "business_diagnostics":
        PROCESSED_DIR / "insight_business_district_diagnostics.csv",

    "business_top10":
        PROCESSED_DIR / "insight_business_district_top10.csv",

    "business_top1":
        PROCESSED_DIR / "insight_business_district_top1.csv",

    "business_overlap":
        PROCESSED_DIR / "insight_business_district_top10_overlap.csv",

    "business_specialization":
        PROCESSED_DIR / "insight_business_district_specialization.csv",

    "business_region":
        PROCESSED_DIR / "insight_business_district_region_summary.csv",

    "weight_sensitivity":
        PROCESSED_DIR / "insight_weight_sensitivity.csv",

    "weight_switchpoints":
        PROCESSED_DIR / "insight_weight_switchpoints.csv",

    "income_sensitivity":
        PROCESSED_DIR / "insight_income_sensitivity.csv",

    "income_switchpoints":
        PROCESSED_DIR / "insight_income_switchpoints.csv",

    "budget_sensitivity":
        PROCESSED_DIR / "insight_budget_sensitivity.csv",

    "budget_effect":
        PROCESSED_DIR / "insight_budget_change_effect.csv",

    "recommendation_stability":
        PROCESSED_DIR / "insight_recommendation_stability.csv",
}

SAVE_OBJECTS = {
    "common_table": common,
    "tradeoff_all_cheaper": cheaper_all,
    "tradeoff_100k_detail": cheaper,
    "tradeoff_summary": tradeoff_summary,
    "tradeoff_regression": tradeoff_regression,
    "housing_band": housing_band_summary,
    "efficiency": efficiency,
    "rank_correlation": rank_correlation,
    "scenario_top10": scenario_top10,
    "business_diagnostics": district_diagnostics,
    "business_top10": business_top10,
    "business_top1": district_top1,
    "business_overlap": district_top10_overlap,
    "business_specialization": district_specialization,
    "business_region": district_region_summary,
    "weight_sensitivity": weight_sensitivity,
    "weight_switchpoints": weight_switchpoints,
    "income_sensitivity": income_sensitivity,
    "income_switchpoints": income_switchpoints,
    "budget_sensitivity": budget_sensitivity,
    "budget_effect": budget_effect,
    "recommendation_stability": recommendation_stability,
}

for key, path in OUTPUTS.items():

    df = SAVE_OBJECTS[key]

    if isinstance(df, pd.DataFrame):
        df.to_csv(
            path,
            index=False,
            encoding="utf-8-sig",
        )

print("[저장 완료]")
for key, path in OUTPUTS.items():
    print(f"- {key}: {path}")


[저장 완료]
- common_table: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_common_work_home_table.csv
- tradeoff_all_cheaper: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_housing_commute_tradeoff_all_cheaper.csv
- tradeoff_100k_detail: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_housing_commute_tradeoff_100k_detail.csv
- tradeoff_summary: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_housing_commute_tradeoff_summary.csv
- tradeoff_regression: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_housing_commute_tradeoff_regression.csv
- housing_band: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_housing_band_commute_summary.csv
- efficiency: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/insight_cost_efficiency_by_work_home.csv
- rank_correlation: /Users/janghwayeong/Desktop/부

## 해석 시 주의

- 주거비와 통근시간의 관계는 **인과관계가 아니라 경향**으로 표현한다.
- 총부담은 시간가치 가정에 따라 달라지므로 단일 숫자를 절대값처럼 해석하지 않는다.
- 소득·가중치·예산 전환점은 실제 데이터에서 계산된 경우에만 결과 문장으로 사용한다.
- 생활비·주거환경 데이터가 현재 공통 테이블에 없다면 임의로 생성하지 않는다.
- 유형화 결과는 추천을 직접 결정하지 않고, 추천 결과의 지역 특성을 설명하는 데 사용한다.